# Avoided GHG vs Supply-Chain GHG Footprint
## Greenings Backtesting Portfolio · Exiobase Tier-0 & Tier-1

**What this notebook shows**  
For each of the 9 Greenings projects and for the portfolio total:
- **Avoided GHG** — self-reported annual avoided emissions (tCO₂e/yr) from `03_monetisation_inputs.csv`
- **Tier-0 SC GHG** — direct supply-chain GHG footprint of the investment (tCO₂e, one-time CAPEX), computed live via `tier0_impact()` with the Exiobase calibration
- **Tier-1 SC GHG** — first upstream round of the supply chain (tCO₂e, one-time CAPEX), computed live via `tier1_impact()` with the Exiobase calibration

> **Unit note:** Avoided GHG is an annual flow (tCO₂e **per year** of operation).  
> Tier-0 and Tier-1 GHG are one-time stocks embedded in the CAPEX transaction.  
> The comparison shows relative scale — not a direct additive accounting.

## Portfolio Overview, Backtesting Methodology & Country Dependency

### Portfolio Composition

The assessment portfolio comprises **nine infrastructure investments** across three sector
archetypes — healthcare facilities (n=3), renewable energy generation (n=3), and rail transport
systems (n=3) — spanning four global regions: Latin America, Sub-Saharan Africa,
South/Southeast Asia, and Western Europe, with a combined committed capital of approximately
**USD 6.2 billion**.

**Greenhouse gas.** The portfolio generates a first- and second-tier supply-chain embodied
footprint of approximately **1.78 MtCO₂e** attributable to construction and procurement
(Tier-0: 1.14 Mt · Tier-1: 0.65 Mt, Exiobase 3.8.1). Six of the nine projects — all
renewable energy and rail investments — additionally report quantified **avoided emissions**
through displacement of fossil-fuel generation or modal shift away from road and air
transport. The three healthcare projects do not record direct avoided GHG, as facility
construction is not classified as a greenhouse-gas-reduction intervention.

**Water.** Water impacts arise through two distinct channels: supply-chain water embedded in
construction materials (concrete, steel, electrical equipment) across all project types, and
operational water intensity associated with reservoir management for hydropower assets. Rail
projects carry the largest supply-chain water footprint owing to high-volume steel and ballast
procurement; healthcare projects reflect moderate embedded water exposure consistent with
general construction activity; hydropower projects additionally face site-specific operational
water use that is not captured in the supply-chain model.

---

### Backtesting Approach

The backtesting methodology applies the current WifOR impact valuation framework
retrospectively to a historical project dataset, using reported CAPEX and self-declared
avoided-emission figures as primary inputs alongside Exiobase 3.8.1 multi-regional
input-output data to compute Tier-0 and Tier-1 supply-chain GHG and water footprints for
each investment. Three carbon-price layers are then benchmarked against one another: the
UBA Methodological Convention 4.0 social cost of carbon (2030 central estimate, PRTP=1%,
412.50 USD/t), WifOR IPCC-aligned scenario coefficients covering six pathways from the
socially optimal baseline to the strict 1.5°C target, and REMIND-MAgPIE country-specific
carbon prices derived from IPCC AR6 WG3 Chapter 3 regional shadow-price data — weighted
by R5 regional multipliers — across all six SSP marker scenarios. The exercise tests the
sensitivity of net impact monetisation — defined as avoided GHG value minus supply-chain
embodied GHG cost — to the choice of carbon pricing framework, establishing that the UBA
reference already exceeds REMIND-MAgPIE scenario prices for every warming pathway above
1.5°C and therefore represents a conservative lower bound for current carbon-damage valuation.

---

### Country Dependency Comparison

The net GHG valuation of the portfolio is structurally determined by country composition,
because the REMIND-MAgPIE R5 regional multipliers assign materially different effective
carbon prices to each host nation: Western Europe (R5OECD, 1.15–1.30×) faces the highest
abatement-cost premium, while Sub-Saharan Africa and Latin America (R5MAF/R5LAM,
0.68–0.88×) receive a discount relative to the global scenario price, reflecting their lower
marginal abatement costs in a cost-optimal carbon market. This regional differentiation
disproportionately penalises the European rail cluster — which accounts for five of nine
projects, 76% of total CAPEX, and the largest supply-chain GHG footprint — whereas the
single large South/Southeast Asian renewable-energy project benefits from a near-global
multiplier (R5ASIA 0.88–0.93×) that still generates the portfolio's only substantial
positive net position (USD 46–3 201 M across SSPs). Under the uniform UBA Central price,
country origin becomes irrelevant and the net portfolio position is driven solely by the
relative size of avoided versus embodied GHG, whereas the REMIND-MAgPIE country-weighted
approach reveals that an identical investment in Germany incurs a supply-chain carbon cost
approximately 60–80% higher than an equivalent investment in Kenya or Ethiopia — a
structural asymmetry that any country-differentiated carbon pricing regime would need to
account for in cross-border portfolio comparisons.

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from IPython.display import display

warnings.filterwarnings("ignore")

# Notebook lives in project_assessment/ — repo root is one level up
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "tvp_dbio"))

from tvp_io_lib import tier0_impact, tier1_impact

FIG_DIR = ROOT / "project_assessment" / "greenings_backtesting"

print("ROOT   :", ROOT)
print("FIG_DIR:", FIG_DIR)
print("tvp_io_lib loaded")

In [ ]:
# ── Project definitions ───────────────────────────────────────────────────────
# Each Greenings project is mapped to the closest Exiobase sector archetype.
# capex_musd is the CAPEX from the Greenings source data (used for both
# avoided-GHG reporting and the Exiobase tier-0/1 computation so the
# investment scale is identical across all three bars).

PROJECTS = [
    # ── Hospitals ──────────────────────────────────────────────────────────
    dict(project_id="HOSP-1",  label="HOSP-1\nHospital · LATAM",    sector="Health_Social",     region="LATAM",   capex_musd=147),
    dict(project_id="HOSP-2",  label="HOSP-2\nHospital · Africa",   sector="Health_Specialized",region="Africa",  capex_musd=250),
    dict(project_id="HOSP-3",  label="HOSP-3\nHospital · Europe",   sector="Health_General",    region="Europe",  capex_musd=150),
    # ── Hydro Power Plants ─────────────────────────────────────────────────
    dict(project_id="HYDRO-1", label="HYDRO-1\nHydro · Africa",     sector="Energy",            region="Africa",  capex_musd=200),
    dict(project_id="HYDRO-2", label="HYDRO-2\nHydro · Asia",       sector="Energy",            region="Asia",    capex_musd=400),
    dict(project_id="HYDRO-3", label="HYDRO-3\nHydro · Europe",     sector="Energy",            region="Europe",  capex_musd=47),
    # ── Rail ───────────────────────────────────────────────────────────────
    dict(project_id="RAIL-1",  label="RAIL-1\nRail Dev · Europe",   sector="Rail_Dev",          region="Europe",  capex_musd=1190),
    dict(project_id="RAIL-2",  label="RAIL-2\nRail Op · Europe",    sector="Rail_Op",           region="Europe",  capex_musd=1350),
    dict(project_id="RAIL-3",  label="RAIL-3\nRail Op · Europe",    sector="Rail_Op",           region="Europe",  capex_musd=2460),
]

print(f"{len(PROJECTS)} projects defined")

In [ ]:
# ── Load avoided GHG from Greenings backtesting ───────────────────────────────
BT_PATH = ROOT / "project_assessment" / "greenings_backtesting" / "03_monetisation_inputs.csv"
bt = pd.read_csv(BT_PATH)[["project_id", "ghg_avoided_tco2e_yr"]]
avoided_ghg: dict = dict(zip(bt.project_id, bt.ghg_avoided_tco2e_yr))

print("Avoided GHG (tCO2e/yr) per project:")
for pid, val in avoided_ghg.items():
    tag = "(no data)" if pd.isna(val) else f"{val:>12,.0f} tCO2e/yr"
    print(f"  {pid:<10} {tag}")

In [ ]:
# ── Compute Tier-0 and Tier-1 GHG via Exiobase ───────────────────────────────
# tier0_impact / tier1_impact are called with database="exiobase".
# Only GHG_tCO2e is extracted — all other stressors are ignored.

rows = []
for p in PROJECTS:
    invest_usd = p["capex_musd"] * 1_000_000
    t0 = tier0_impact(invest_usd, p["sector"], p["region"], database="exiobase")
    t1 = tier1_impact(invest_usd, p["sector"], p["region"], database="exiobase")
    rows.append({
        "project_id":         p["project_id"],
        "label":              p["label"],
        "sector":             p["sector"],
        "region":             p["region"],
        "capex_musd":         p["capex_musd"],
        "avoided_ghg_tco2e":  avoided_ghg.get(p["project_id"], np.nan),
        "tier0_ghg_tco2e":    t0["GHG_tCO2e"],
        "tier1_ghg_tco2e":    t1["GHG_tCO2e"],
        "t0_backend":         t0["backend"],
    })
    print(f"  {p['project_id']:<10}  T0={t0['GHG_tCO2e']:>10,.0f}  T1={t1['GHG_tCO2e']:>10,.0f}  [{t0['backend']}]")

df = pd.DataFrame(rows)
df["tier01_ghg_tco2e"] = df["tier0_ghg_tco2e"] + df["tier1_ghg_tco2e"]

# ── Portfolio totals (sum across all projects; avoided NaN treated as 0) ─────
portfolio = {
    "project_id":        "PORTFOLIO",
    "label":             "Portfolio\nTotal",
    "capex_musd":        df["capex_musd"].sum(),
    "avoided_ghg_tco2e": df["avoided_ghg_tco2e"].fillna(0).sum(),
    "tier0_ghg_tco2e":   df["tier0_ghg_tco2e"].sum(),
    "tier1_ghg_tco2e":   df["tier1_ghg_tco2e"].sum(),
    "tier01_ghg_tco2e":  df["tier01_ghg_tco2e"].sum(),
}
print(f"\n  {'PORTFOLIO':<10}  T0={portfolio['tier0_ghg_tco2e']:>10,.0f}  T1={portfolio['tier1_ghg_tco2e']:>10,.0f}")

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
summary = df[[
    "project_id", "sector", "region", "capex_musd",
    "avoided_ghg_tco2e", "tier0_ghg_tco2e", "tier1_ghg_tco2e", "tier01_ghg_tco2e"
]].copy()

summary.columns = [
    "Project", "Sector", "Region", "CAPEX (MUSD)",
    "Avoided GHG (tCO2e/yr)", "Tier-0 SC GHG (tCO2e)",
    "Tier-1 SC GHG (tCO2e)", "Tier-0+1 SC GHG (tCO2e)"
]

# Append portfolio total row
port_row = pd.DataFrame([{
    "Project":                 "PORTFOLIO",
    "Sector":                  "—",
    "Region":                  "—",
    "CAPEX (MUSD)":            portfolio["capex_musd"],
    "Avoided GHG (tCO2e/yr)": portfolio["avoided_ghg_tco2e"],
    "Tier-0 SC GHG (tCO2e)":  portfolio["tier0_ghg_tco2e"],
    "Tier-1 SC GHG (tCO2e)":  portfolio["tier1_ghg_tco2e"],
    "Tier-0+1 SC GHG (tCO2e)": portfolio["tier01_ghg_tco2e"],
}])
summary = pd.concat([summary, port_row], ignore_index=True)

display(
    summary.style
    .format({
        "CAPEX (MUSD)":             "{:,.0f}",
        "Avoided GHG (tCO2e/yr)":  lambda v: f"{v:,.0f}" if pd.notna(v) and v > 0 else "—",
        "Tier-0 SC GHG (tCO2e)":  "{:,.0f}",
        "Tier-1 SC GHG (tCO2e)":  "{:,.0f}",
        "Tier-0+1 SC GHG (tCO2e)": "{:,.0f}",
    })
    .set_caption("GHG comparison · Exiobase calibrated · all values in tCO₂e")
    .highlight_null(color="#f0f0f0")
    .apply(lambda col: ["font-weight:bold; background:#eaf4ff" if v == "PORTFOLIO" else "" for v in col],
           subset=["Project"])
)

## Per-Project Comparison

In [ ]:
C_AVOIDED = "#27ae60"
C_TIER0   = "#c0392b"
C_TIER1   = "#e67e22"

BAR_LABELS = ["Avoided GHG\n(tCO₂e/yr)", "Tier-0 SC\n(tCO₂e)", "Tier-1 SC\n(tCO₂e)"]
BAR_COLORS = [C_AVOIDED, C_TIER0, C_TIER1]

def fmt_k(x):
    if abs(x) >= 1_000_000:
        return f"{x/1e6:.1f}M"
    if abs(x) >= 1_000:
        return f"{x/1e3:.0f}k"
    return f"{x:.0f}"

fig, axes = plt.subplots(3, 3, figsize=(15, 13))
fig.suptitle(
    "Avoided GHG vs Supply-Chain GHG · Per Project (Exiobase calibration)",
    fontsize=13, fontweight="bold", y=1.01
)

for ax, (_, row) in zip(axes.flatten(), df.iterrows()):
    avoided_val  = row["avoided_ghg_tco2e"]
    avoided_plot = 0.0 if pd.isna(avoided_val) else float(avoided_val)
    vals = [avoided_plot, row["tier0_ghg_tco2e"], row["tier1_ghg_tco2e"]]

    bars  = ax.bar(BAR_LABELS, vals, color=BAR_COLORS, width=0.55, edgecolor="white", linewidth=0.8)
    y_max = max(vals) if max(vals) > 0 else 1

    for bar, val in zip(bars, vals):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, val + y_max * 0.02,
                    f"{val:,.0f}", ha="center", va="bottom", fontsize=7.5, fontweight="bold")

    if pd.isna(avoided_val):
        ax.text(0, y_max * 0.35, "no GHG\nreported",
                ha="center", va="center", fontsize=7, color="#888", style="italic")

    ax.set_title(row["label"], fontsize=9, fontweight="bold", pad=4)
    ax.set_ylim(0, y_max * 1.25)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: fmt_k(x)))
    ax.tick_params(axis="x", labelsize=7.5)
    ax.tick_params(axis="y", labelsize=7.5)
    ax.spines[["top", "right"]].set_visible(False)
    ax.annotate(f"CAPEX: {row['capex_musd']:,} MUSD",
                xy=(0.97, 0.97), xycoords="axes fraction",
                ha="right", va="top", fontsize=7, color="#555")

plt.tight_layout()
plt.savefig(FIG_DIR / "fig_ghg_avoided_vs_sc_per_project.png", dpi=130, bbox_inches="tight")
plt.show()

## Portfolio Total

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

port_vals = [
    portfolio["avoided_ghg_tco2e"],
    portfolio["tier0_ghg_tco2e"],
    portfolio["tier1_ghg_tco2e"],
]
bars  = ax.bar(BAR_LABELS, port_vals, color=BAR_COLORS, width=0.45, edgecolor="white", linewidth=1)
y_max = max(port_vals)

for bar, val in zip(bars, port_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, val + y_max * 0.01,
            f"{val:,.0f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

ax.set_title(
    f"Portfolio Total  ·  9 projects  ·  CAPEX {portfolio['capex_musd']:,.0f} MUSD",
    fontsize=11, fontweight="bold", pad=8
)
ax.set_ylabel("tCO₂e", fontsize=10)
ax.set_ylim(0, y_max * 1.18)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: fmt_k(x)))
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(axis="x", labelsize=9)

ratio_t0  = portfolio["avoided_ghg_tco2e"] / portfolio["tier0_ghg_tco2e"]
ratio_t01 = portfolio["avoided_ghg_tco2e"] / portfolio["tier01_ghg_tco2e"]
ax.annotate(
    f"Avoided / Tier-0: {ratio_t0:.1f}×\nAvoided / Tier-0+1: {ratio_t01:.1f}×",
    xy=(0.97, 0.95), xycoords="axes fraction", ha="right", va="top", fontsize=9,
    bbox=dict(boxstyle="round,pad=0.3", fc="#f8f8f8", ec="#ccc")
)
ax.annotate(
    "* Avoided GHG = annual operational flow (tCO₂e/yr)\n"
    "  Tier-0/1 SC  = one-time CAPEX footprint (tCO₂e)\n"
    "  Hospitals excluded from avoided GHG sum (no data)",
    xy=(0.01, -0.18), xycoords="axes fraction",
    ha="left", va="bottom", fontsize=7.5, color="#666", style="italic"
)

plt.tight_layout()
plt.savefig(FIG_DIR / "fig_ghg_avoided_vs_sc_portfolio.png", dpi=130, bbox_inches="tight")
plt.show()

## Stacked View — Tier-0 + Tier-1 by Project

In [ ]:
labels_h     = [r["project_id"] + "\n" + r["sector"] for _, r in df.iterrows()] + ["PORTFOLIO"]
avoided_vals = list(df["avoided_ghg_tco2e"].fillna(0))  + [portfolio["avoided_ghg_tco2e"]]
tier0_vals   = list(df["tier0_ghg_tco2e"])               + [portfolio["tier0_ghg_tco2e"]]
tier1_vals   = list(df["tier1_ghg_tco2e"])               + [portfolio["tier1_ghg_tco2e"]]

y = np.arange(len(labels_h))
h = 0.28

fig, ax = plt.subplots(figsize=(12, 7))

ax.barh(y + h, avoided_vals, height=h, color=C_AVOIDED, label="Avoided GHG (tCO₂e/yr)")
ax.barh(y,     tier0_vals,   height=h, color=C_TIER0,   label="Tier-0 SC GHG (tCO₂e)")
ax.barh(y,     tier1_vals,   height=h, color=C_TIER1,   left=tier0_vals, label="Tier-1 SC GHG (tCO₂e)")

ax.axhline(len(labels_h) - 1.5, color="#aaa", linewidth=0.8, linestyle="--")

ax.set_yticks(y + h / 2)
ax.set_yticklabels(labels_h, fontsize=8)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: fmt_k(x)))
ax.set_xlabel("tCO₂e", fontsize=10)
ax.set_title(
    "Avoided GHG vs Tier-0 + Tier-1 Supply-Chain GHG · Exiobase · All Projects + Portfolio",
    fontsize=11, fontweight="bold", pad=10
)
ax.legend(loc="lower right", fontsize=8, framealpha=0.9)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(FIG_DIR / "fig_ghg_avoided_vs_sc_stacked.png", dpi=130, bbox_inches="tight")
plt.show()

## Monetised GHG — WifOR Value Factor (most recent: 2024-11-18)

GHG physical quantities are converted to USD using the most recent WifOR coefficient file.  
Three scenarios are shown as a range:

| Scenario | Coefficient | Source |
|----------|-------------|--------|
| **GHG_BASE** | 63.70 USD/tCO₂e | Nordhaus DICE — calibrated baseline (used in main assessment) |
| **GHG_PARIS** | 58.65 USD/tCO₂e | Paris-consistent update — lower due to tech-cost trajectory |
| **GHG_T<2** | 73.01 USD/tCO₂e | 2°C target damage — upper bound in this comparison |

The coefficient is **globally uniform** in this WifOR release — country and sector do not shift the GHG social cost.  
Unit note: avoided GHG remains an annual flow (MUSD/yr); supply-chain GHG is a one-time CAPEX cost (MUSD).

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")

# ── Load GHG coefficients from the most recent WifOR file ────────────────────
VF_PATH = ROOT / "data" / "value_factors" / "2024-11-18_formatted_MonGHG_my.h5"
YEAR    = "2019"

with pd.HDFStore(str(VF_PATH), "r") as store:
    ghg_df = store["/coefficient"]

def get_coeff_usd_per_t(indicator: str) -> float:
    """Read globally-uniform coefficient; convert USD/kg → USD/tCO2e."""
    return abs(float(ghg_df.loc[(YEAR, indicator)].iloc[0])) * 1_000

C_BASE  = get_coeff_usd_per_t("COEFFICIENT GHG_BASE, in USD (WifOR)")
C_PARIS = get_coeff_usd_per_t("COEFFICIENT GHG_PARIS_UPDATE, in USD (WifOR)")
C_T2    = get_coeff_usd_per_t("COEFFICIENT GHG_T<2, in USD (WifOR)")

print(f"Source : {VF_PATH.name}")
print(f"Year   : {YEAR}")
print(f"GHG_BASE   = {C_BASE:.2f} USD/tCO2e  (Nordhaus DICE)")
print(f"GHG_PARIS  = {C_PARIS:.2f} USD/tCO2e  (Paris-consistent)")
print(f"GHG_T<2    = {C_T2:.2f} USD/tCO2e  (2°C target)")

# ── Monetise: tCO2e → MUSD ───────────────────────────────────────────────────
def to_musd(tco2e: float, coeff: float) -> float:
    return tco2e * coeff / 1_000_000

mon_rows = []
for _, row in df.iterrows():
    av   = 0.0 if pd.isna(row["avoided_ghg_tco2e"]) else float(row["avoided_ghg_tco2e"])
    t0   = float(row["tier0_ghg_tco2e"])
    t1   = float(row["tier1_ghg_tco2e"])
    mon_rows.append({
        "project_id":         row["project_id"],
        "label":              row["label"],
        "has_avoided":        not pd.isna(row["avoided_ghg_tco2e"]),
        # physical
        "avoided_tco2e":      av,
        "tier0_tco2e":        t0,
        "tier1_tco2e":        t1,
        # monetised BASE
        "av_musd_base":       to_musd(av, C_BASE),
        "t0_musd_base":       to_musd(t0, C_BASE),
        "t1_musd_base":       to_musd(t1, C_BASE),
        # monetised PARIS
        "av_musd_paris":      to_musd(av, C_PARIS),
        "t0_musd_paris":      to_musd(t0, C_PARIS),
        "t1_musd_paris":      to_musd(t1, C_PARIS),
        # monetised T<2
        "av_musd_t2":         to_musd(av, C_T2),
        "t0_musd_t2":         to_musd(t0, C_T2),
        "t1_musd_t2":         to_musd(t1, C_T2),
    })

mdf = pd.DataFrame(mon_rows)

# Portfolio totals
port_m = pd.Series({k: mdf[k].sum() if k not in ("project_id","label","has_avoided") else None
                    for k in mdf.columns})
port_m["project_id"] = "PORTFOLIO"
port_m["label"]      = "Portfolio\nTotal"
port_m["has_avoided"] = True

# Print summary
print("\nMonetised GHG summary (BASE, MUSD):")
print(mdf[["project_id","av_musd_base","t0_musd_base","t1_musd_base"]].round(2).to_string(index=False))
print(f"\n{'PORTFOLIO':<15} av={port_m.av_musd_base:.2f}  t0={port_m.t0_musd_base:.2f}  t1={port_m.t1_musd_base:.2f}")

In [ ]:
MON_LABELS = ["Avoided GHG\n(MUSD/yr)", "Tier-0 SC\n(MUSD)", "Tier-1 SC\n(MUSD)"]

fig, axes = plt.subplots(3, 3, figsize=(15, 13))
fig.suptitle(
    f"Monetised GHG · WifOR GHG_BASE ({C_BASE:.1f} USD/tCO₂e · Nordhaus DICE 2019)\n"
    f"Error bars: GHG_PARIS ({C_PARIS:.1f}) – GHG_T<2 ({C_T2:.1f} USD/tCO₂e)",
    fontsize=11, fontweight="bold", y=1.02
)

for ax, (_, row) in zip(axes.flatten(), mdf.iterrows()):
    base_vals  = [row["av_musd_base"],  row["t0_musd_base"],  row["t1_musd_base"]]
    paris_vals = [row["av_musd_paris"], row["t0_musd_paris"], row["t1_musd_paris"]]
    t2_vals    = [row["av_musd_t2"],    row["t0_musd_t2"],    row["t1_musd_t2"]]

    x     = np.arange(len(MON_LABELS))
    bars  = ax.bar(x, base_vals, color=BAR_COLORS, width=0.55,
                   edgecolor="white", linewidth=0.8)

    # Range bars: PARIS (lower) to T<2 (upper)
    yerr_lo = [b - p for b, p in zip(base_vals, paris_vals)]
    yerr_hi = [t - b for t, b in zip(t2_vals, base_vals)]
    ax.errorbar(x, base_vals,
                yerr=[yerr_lo, yerr_hi],
                fmt="none", color="#333", capsize=4, capthick=1.2, lw=1.2, zorder=5)

    y_max = max(t2_vals) if max(t2_vals) > 0 else 1
    for bar, val in zip(bars, base_vals):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, val + y_max * 0.03,
                    f"{val:.2f}", ha="center", va="bottom", fontsize=7, fontweight="bold")

    if not row["has_avoided"]:
        ax.text(0, y_max * 0.35, "no GHG\nreported",
                ha="center", va="center", fontsize=7, color="#888", style="italic")

    ax.set_xticks(x)
    ax.set_xticklabels(MON_LABELS, fontsize=7.5)
    ax.set_title(row["label"], fontsize=9, fontweight="bold", pad=4)
    ax.set_ylim(0, y_max * 1.3)
    ax.set_ylabel("MUSD", fontsize=7)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.1f}"))
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(axis="y", labelsize=7)

plt.tight_layout()
plt.savefig(FIG_DIR / "fig_ghg_monetised_per_project.png", dpi=130, bbox_inches="tight")
plt.show()

In [ ]:
# ── Portfolio total — scenario comparison ────────────────────────────────────
scenarios = {
    f"GHG_BASE\n{C_BASE:.0f} USD/t":   ("av_musd_base",  "t0_musd_base",  "t1_musd_base"),
    f"GHG_PARIS\n{C_PARIS:.0f} USD/t": ("av_musd_paris", "t0_musd_paris", "t1_musd_paris"),
    f"GHG_T<2\n{C_T2:.0f} USD/t":      ("av_musd_t2",    "t0_musd_t2",    "t1_musd_t2"),
}

n_scen = len(scenarios)
n_bars = 3   # avoided, t0, t1
x      = np.arange(n_scen)
w      = 0.22

fig, ax = plt.subplots(figsize=(10, 6))

offsets = [-w, 0, w]
for idx, (bar_label, color) in enumerate(zip(
        ["Avoided GHG (MUSD/yr)", "Tier-0 SC (MUSD)", "Tier-1 SC (MUSD)"], BAR_COLORS)):
    vals = [mdf[cols[idx]].sum() for cols in scenarios.values()]
    bars = ax.bar(x + offsets[idx], vals, width=w, color=color,
                  label=bar_label, edgecolor="white", linewidth=0.8)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + ax.get_ylim()[1] * 0.005 if ax.get_ylim()[1] > 0 else val + 1,
                f"{val:.1f}", ha="center", va="bottom", fontsize=8, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(list(scenarios.keys()), fontsize=9)
ax.set_ylabel("MUSD", fontsize=10)
ax.set_title(
    f"Portfolio Total — Monetised GHG · Three WifOR Scenarios\n"
    f"(9 projects · CAPEX {df['capex_musd'].sum():,.0f} MUSD"
    f" · Hospitals excluded from avoided GHG)",
    fontsize=11, fontweight="bold", pad=8
)
ax.legend(fontsize=9, framealpha=0.9, loc="upper right")
ax.spines[["top", "right"]].set_visible(False)

# Ratio annotation for BASE scenario
av_b  = mdf["av_musd_base"].sum()
sc_b  = mdf["t0_musd_base"].sum() + mdf["t1_musd_base"].sum()
ax.annotate(
    f"BASE: Avoided / (T0+T1) = {av_b/sc_b:.1f}×",
    xy=(0.02, 0.97), xycoords="axes fraction",
    ha="left", va="top", fontsize=9,
    bbox=dict(boxstyle="round,pad=0.3", fc="#f8f8f8", ec="#ccc")
)

plt.tight_layout()
plt.savefig(FIG_DIR / "fig_ghg_monetised_portfolio.png", dpi=130, bbox_inches="tight")
plt.show()

## IPCC 2030 Country-Specific Monetisation — PARIS_UPDATE & T<2

GHG value factors are extracted for **year 2030** from the most recent WifOR file
(`2024-11-18_formatted_MonGHG_my.h5`), looking up each project's **representative ISO3 country
and NACE sector**.

Two of the **six main IPCC-aligned scenarios** are selected for this analysis:

| # | Scenario | 2030 coeff. | IPCC alignment |
|---|---|---|---|
| 1 | GHG_BASE | 104.73 USD/tCO₂e | Nordhaus DICE — calibrated baseline |
| 2 | GHG_OPTIMAL | 78.71 USD/tCO₂e | Cost-optimal mitigation path |
| 3 | **GHG_PARIS_UPDATE ◄** | **96.20 USD/tCO₂e** | Paris Agreement — well below 2 °C |
| 4 | **GHG_T<2 ◄** | **121.05 USD/tCO₂e** | IPCC AR6 SSP1-2.6 — 2 °C carbon budget |
| 5 | GHG_T<1.5 | 5 736 USD/tCO₂e | IPCC AR6 SSP1-1.9 — 1.5 °C (excluded: extreme scale) |
| 6 | GHG_ALTDAM | 191.14 USD/tCO₂e | Alternative damage function |

> 2030 coefficients are **+64 % (PARIS)** and **+66 % (T<2)** above their 2019 values,
> reflecting the steepening social cost of carbon along IPCC pathways.

### Country & NACE mapping per project

| Project | Country | ISO3 | NACE | Sector |
|---|---|---|---|---|
| HOSP-1 | Colombia | COL | Q | Health & Social |
| HOSP-2 | Kenya | KEN | Q | Health Specialized |
| HOSP-3 | Germany | DEU | Q | Health General |
| HYDRO-1 | Ethiopia | ETH | D35 | Electricity supply |
| HYDRO-2 | India | IND | D35 | Electricity supply |
| HYDRO-3 | Germany | DEU | D35 | Electricity supply |
| RAIL-1 | Germany | DEU | F | Construction |
| RAIL-2 | Germany | DEU | H49 | Land transport |
| RAIL-3 | Germany | DEU | H49 | Land transport |

> **Note:** The WifOR GHG social cost is globally uniform in this release.
> Country-specific GHG intensity differences are captured through the Exiobase
> tier-0/1 physical quantities (tCO₂e per MUSD invested), which vary by region.


In [ ]:
# ── IPCC 2030 country-specific monetisation — load data ──────────────────────
YEAR_30    = "2030"
SCEN_PARIS = "COEFFICIENT GHG_PARIS_UPDATE, in USD (WifOR)"
SCEN_T2    = "COEFFICIENT GHG_T<2, in USD (WifOR)"

# Representative country and NACE sector per project
PROJ_EXT = {
    "HOSP-1":  dict(iso3="COL", nace="Q",   country="Colombia"),
    "HOSP-2":  dict(iso3="KEN", nace="Q",   country="Kenya"),
    "HOSP-3":  dict(iso3="DEU", nace="Q",   country="Germany"),
    "HYDRO-1": dict(iso3="ETH", nace="D35", country="Ethiopia"),
    "HYDRO-2": dict(iso3="IND", nace="D35", country="India"),
    "HYDRO-3": dict(iso3="DEU", nace="D35", country="Germany"),
    "RAIL-1":  dict(iso3="DEU", nace="F",   country="Germany"),
    "RAIL-2":  dict(iso3="DEU", nace="H49", country="Germany"),
    "RAIL-3":  dict(iso3="DEU", nace="H49", country="Germany"),
}

with pd.HDFStore(str(VF_PATH), "r") as store:
    _ghg30 = store["/coefficient"]

def coeff_2030(indicator: str, iso3: str, nace: str) -> float:
    """USD/kg → USD/tCO2e, country + NACE lookup at year 2030."""
    return abs(float(_ghg30.loc[(YEAR_30, indicator), (iso3, nace)])) * 1_000

# Print all 6 main IPCC-aligned scenario coefficients at 2030
SCEN_TABLE = [
    ("GHG_BASE",         "COEFFICIENT GHG_BASE, in USD (WifOR)"),
    ("GHG_OPTIMAL",      "COEFFICIENT GHG_OPTIMAL, in USD (WifOR)"),
    ("GHG_PARIS_UPDATE", SCEN_PARIS),
    ("GHG_T<2",          SCEN_T2),
    ("GHG_T<1.5",        "COEFFICIENT GHG_T<1.5, in USD (WifOR)"),
    ("GHG_ALTDAM",       "COEFFICIENT GHG_ALTDAM, in USD (WifOR)"),
]
print(f"{'Scenario':<22}  {'2030 coeff (USD/tCO₂e)':>22}")
print("─" * 50)
for name, key in SCEN_TABLE:
    c    = coeff_2030(key, "DEU", "Q")
    star = "  ◄ selected" if name in ("GHG_PARIS_UPDATE", "GHG_T<2") else ""
    print(f"  {name:<22}  {c:>18.2f}{star}")

# Build 2030 monetised DataFrame
rows_30 = []
for _, row in df.iterrows():
    pid  = row["project_id"]
    ext  = PROJ_EXT[pid]
    iso3 = ext["iso3"]
    nace = ext["nace"]

    c_p  = coeff_2030(SCEN_PARIS, iso3, nace)
    c_t2 = coeff_2030(SCEN_T2,    iso3, nace)

    av = 0.0 if pd.isna(row["avoided_ghg_tco2e"]) else float(row["avoided_ghg_tco2e"])
    sc = float(row["tier0_ghg_tco2e"]) + float(row["tier1_ghg_tco2e"])

    rows_30.append({
        "project_id":    pid,
        "label":         row["label"],
        "iso3":          iso3,
        "nace":          nace,
        "country":       ext["country"],
        "has_avoided":   not pd.isna(row["avoided_ghg_tco2e"]),
        "avoided_tco2e": av,
        "sc_tco2e":      sc,
        "c_paris":       c_p,
        "c_t2":          c_t2,
        # PARIS 2030
        "av_p30":  to_musd(av, c_p),
        "sc_p30":  to_musd(sc, c_p),
        "net_p30": to_musd(av - sc, c_p),
        # T<2 2030
        "av_t2":   to_musd(av, c_t2),
        "sc_t2":   to_musd(sc, c_t2),
        "net_t2":  to_musd(av - sc, c_t2),
    })

mdf30 = pd.DataFrame(rows_30)

print(f"\n{'Project':<10} {'Iso3':<5} {'Av_P':>8} {'SC_P':>8} {'Net_P':>8} | {'Av_T2':>8} {'SC_T2':>8} {'Net_T2':>8}")
print("─" * 80)
for _, r in mdf30.iterrows():
    flag = "✓" if r.has_avoided else "—"
    print(f"  {r.project_id:<10} {r.iso3:<5} {r.av_p30:>8.2f} {r.sc_p30:>8.2f} {r.net_p30:>8.2f} | {r.av_t2:>8.2f} {r.sc_t2:>8.2f} {r.net_t2:>8.2f}  {flag}")
print(f"  {'PORTFOLIO':<10} {'':>5} {mdf30.av_p30.sum():>8.2f} {mdf30.sc_p30.sum():>8.2f} {mdf30.net_p30.sum():>8.2f} | {mdf30.av_t2.sum():>8.2f} {mdf30.sc_t2.sum():>8.2f} {mdf30.net_t2.sum():>8.2f}")


In [ ]:
# ── Per-project: Avoided vs Supply-chain @ IPCC 2030 scenarios ───────────────
from matplotlib.patches import Patch

_SCENS = [
    ("PARIS 2030\n96.2 USD/t",  "av_p30", "sc_p30", "net_p30"),
    ("T<2 2030\n121.0 USD/t",   "av_t2",  "sc_t2",  "net_t2"),
]
_C_AV = "#27ae60"
_C_SC = "#c0392b"
_C_NP = "#2980b9"   # net positive
_C_NN = "#e74c3c"   # net negative

fig, axes = plt.subplots(3, 3, figsize=(16, 13))
fig.suptitle(
    "Per-Project IPCC 2030 Monetised GHG · PARIS_UPDATE vs T<2\n"
    "Green = Avoided  ·  Red = Supply-Chain T0+T1  ·  Blue/Red = Net Delta  ·  Country × NACE specific",
    fontsize=11, fontweight="bold", y=1.02
)

for ax, (_, row) in zip(axes.flatten(), mdf30.iterrows()):
    all_vals = [row[c] for _, av_c, sc_c, net_c in _SCENS for c in (av_c, sc_c, net_c)]
    y_span   = max(abs(v) for v in all_vals) if any(abs(v) > 0 for v in all_vals) else 1.0

    xs = np.arange(len(_SCENS))
    w  = 0.22

    for i, (_, av_c, sc_c, net_c) in enumerate(_SCENS):
        av_v  = row[av_c]
        sc_v  = row[sc_c]
        net_v = row[net_c]
        nc    = _C_NP if net_v >= 0 else _C_NN

        ax.bar(xs[i] - w, av_v,  width=w, color=_C_AV, edgecolor="white", lw=0.7)
        ax.bar(xs[i],     sc_v,  width=w, color=_C_SC, edgecolor="white", lw=0.7)
        ax.bar(xs[i] + w, net_v, width=w, color=nc,    edgecolor="white", lw=0.7, alpha=0.85)

        for xpos, val in [(xs[i] - w, av_v), (xs[i], sc_v), (xs[i] + w, net_v)]:
            if abs(val) > y_span * 0.02:
                off = y_span * 0.07
                ax.text(xpos, val + (off if val >= 0 else -off),
                        f"{val:.1f}", ha="center",
                        va="bottom" if val >= 0 else "top",
                        fontsize=6.5, fontweight="bold")

    ax.axhline(0, color="#666", lw=0.8, ls="--", zorder=0)
    ax.set_xticks(xs)
    ax.set_xticklabels([s[0] for s in _SCENS], fontsize=7.5)
    ax.set_title(
        f"{row['label']}\n{row['country']} ({row['iso3']}) · NACE {row['nace']}",
        fontsize=8.5, fontweight="bold", pad=4
    )
    ax.set_ylabel("MUSD", fontsize=7)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.0f}"))
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(labelsize=7)
    ax.set_ylim(-y_span * 1.45, y_span * 1.45)

    if not row["has_avoided"]:
        ax.text(0.5, 0.78, "no avoided GHG\nreported",
                transform=ax.transAxes, ha="center", va="center",
                fontsize=7.5, color="#999", style="italic")

leg_handles = [
    Patch(facecolor=_C_AV, label="Avoided GHG (MUSD/yr)"),
    Patch(facecolor=_C_SC, label="Supply-Chain T0+T1 (MUSD)"),
    Patch(facecolor=_C_NP, label="Net (+): Avoided > SC"),
    Patch(facecolor=_C_NN, label="Net (−): SC > Avoided"),
]
fig.legend(handles=leg_handles, loc="lower center", ncol=4, fontsize=9,
           framealpha=0.9, bbox_to_anchor=(0.5, -0.02))

plt.tight_layout()
plt.savefig(FIG_DIR / "fig_ghg_ipcc2030_per_project.png", dpi=130, bbox_inches="tight")
plt.show()


In [ ]:
# ── Portfolio total — IPCC 2030 monetised GHG (PARIS_UPDATE vs T<2) ──────────
_C_AV30  = "#27ae60"
_C_SC30  = "#c0392b"
_C_NET30 = "#2980b9"

fig, ax = plt.subplots(figsize=(11, 7))

SCENS_30 = [
    ("PARIS_UPDATE 2030\n(96.2 USD/tCO₂e)",  "av_p30", "sc_p30", "net_p30"),
    ("GHG_T<2 2030\n(121.0 USD/tCO₂e)",      "av_t2",  "sc_t2",  "net_t2"),
]
bar_defs30 = [
    ("Avoided GHG (MUSD/yr)",      _C_AV30,  -0.22),
    ("Supply-Chain T0+T1 (MUSD)",  _C_SC30,   0.00),
    ("Net = Avoided − SC (MUSD)",  _C_NET30,  0.22),
]

xs30 = np.arange(len(SCENS_30))

for bar_idx, (bar_label, bcolor, offset) in enumerate(bar_defs30):
    col_keys = [s[bar_idx + 1] for s in SCENS_30]
    vals     = [mdf30[ck].sum() for ck in col_keys]
    bars30   = ax.bar(xs30 + offset, vals, width=0.20,
                      color=bcolor, label=bar_label,
                      edgecolor="white", linewidth=0.8)
    for bar, val in zip(bars30, vals):
        va  = "bottom" if val >= 0 else "top"
        off = max(abs(val) * 0.04, 5)
        ax.text(bar.get_x() + bar.get_width() / 2, val + (off if val >= 0 else -off),
                f"{val:.1f}", ha="center", va=va, fontsize=9, fontweight="bold")

ax.axhline(0, color="#555", lw=0.9, ls="--")
ax.set_xticks(xs30)
ax.set_xticklabels([s[0] for s in SCENS_30], fontsize=10)
ax.set_ylabel("MUSD", fontsize=11)
ax.set_title(
    "Portfolio Total — IPCC 2030 Monetised GHG\n"
    f"PARIS_UPDATE vs GHG_T<2  ·  9 projects  ·  CAPEX {df['capex_musd'].sum():,.0f} MUSD",
    fontsize=11, fontweight="bold", pad=8
)
ax.legend(fontsize=9, loc="lower right", framealpha=0.9)
ax.spines[["top", "right"]].set_visible(False)

av_p_tot  = mdf30["av_p30"].sum()
sc_p_tot  = mdf30["sc_p30"].sum()
av_t2_tot = mdf30["av_t2"].sum()
sc_t2_tot = mdf30["sc_t2"].sum()
info = (
    f"PARIS 2030:  Avoided/SC = {av_p_tot/sc_p_tot:.2f}×   Net = {av_p_tot - sc_p_tot:.1f} MUSD\n"
    f"T<2   2030:  Avoided/SC = {av_t2_tot/sc_t2_tot:.2f}×   Net = {av_t2_tot - sc_t2_tot:.1f} MUSD\n"
    f"2030 vs 2019 price uplift:  PARIS +{(96.20/58.65-1)*100:.0f}%   T<2 +{(121.05/73.01-1)*100:.0f}%"
)
ax.annotate(info, xy=(0.02, 0.97), xycoords="axes fraction",
            ha="left", va="top", fontsize=9,
            bbox=dict(boxstyle="round,pad=0.4", fc="#f4f8fc", ec="#aaccee"))

plt.tight_layout()
plt.savefig(FIG_DIR / "fig_ghg_ipcc2030_portfolio.png", dpi=130, bbox_inches="tight")
plt.show()


## UBA Value Factor Assessment — All Portfolio Countries

GHG physical quantities are monetised using the **German Federal Environment Agency (UBA)  
Methodological Convention 4.0** (December 2025), the regulatory reference standard for the EU.

### UBA GHG scenarios (year 2030)

| Scenario | EUR_2025/t | USD/t (×1.10) | Basis |
|---|---|---|---|
| **Central (PRTP=1%)** | 375 EUR/t | **412.50 USD/t** | Recommended rate — moderate discounting |
| **High (PRTP=0%)** | 1 050 EUR/t | **1 155.00 USD/t** | Equity-weighted — zero pure-time preference |

### Cross-system comparison @ 2030

| System | USD/tCO₂e | Premium vs WifOR PARIS |
|---|---|---|
| WifOR PARIS_UPDATE | 96.20 | — |
| WifOR T<2 | 121.05 | +26 % |
| **UBA Central (PRTP=1%)** | **412.50** | **+329 %** |
| **UBA High (PRTP=0%)** | **1 155.00** | **+1 100 %** |

> UBA factors reflect **full damage costs** (health, biodiversity, material),
> not mitigation costs. This explains the 3–10× premium over WifOR IPCC trajectory pricing.

### Portfolio countries covered
Colombia (COL) · Kenya (KEN) · Germany (DEU) · Ethiopia (ETH) · India (IND)

UBA applies a **globally-uniform** carbon price — the same per-tonne rate for all countries.  
Country-specific GHG intensity differences are captured through Exiobase tier-0/1 quantities  
(tCO₂e per MUSD invested), which vary by region.


In [ ]:
# ── Load UBA value factors and compute monetised values ──────────────────────
import os

# Locate UBA output directory
_UBA_CANDIDATES = [
    Path("/home/deuler/transitionvaluation/uba-value-factors/output"),
    Path("/home/deuler/tvp4/uba-value-factors/output"),
    ROOT / "transitionvaluation" / "uba-value-factors" / "output",
]
UBA_DIR = next((p for p in _UBA_CANDIDATES if p.is_dir() and (p / "01_uba4_ghg.csv").exists()), None)
if UBA_DIR is None:
    raise FileNotFoundError("UBA output directory not found")
print(f"UBA dir : {UBA_DIR}")

EUR2025_TO_USD = 1.10   # indicative EUR_2025 → USD rate

uba_ghg = pd.read_csv(UBA_DIR / "01_uba4_ghg.csv")

# 2030, CO2_CO2eq only
g30 = uba_ghg[(uba_ghg.emission_year == 2030) & (uba_ghg.substance == "CO2_CO2eq")]
UBA_C_EUPT = float(g30[g30.prtp_pct == 1]["value_factor"].iloc[0])   # central PRTP=1%
UBA_H_EUPT = float(g30[g30.prtp_pct == 0]["value_factor"].iloc[0])   # high PRTP=0%
UBA_C_USD  = UBA_C_EUPT * EUR2025_TO_USD
UBA_H_USD  = UBA_H_EUPT * EUR2025_TO_USD

print(f"\nUBA GHG @ 2030 (CO2_CO2eq):")
print(f"  Central PRTP=1% : {UBA_C_EUPT:>6} EUR_2025/t  =>  {UBA_C_USD:>8.2f} USD/t")
print(f"  High    PRTP=0% : {UBA_H_EUPT:>6} EUR_2025/t  =>  {UBA_H_USD:>8.2f} USD/t")

# WifOR 2030 for comparison
C_PARIS_30 = mdf30["c_paris"].iloc[0]
C_T2_30    = mdf30["c_t2"].iloc[0]
print(f"\nWifOR 2030 for comparison:")
print(f"  PARIS_UPDATE     :                      {C_PARIS_30:>8.2f} USD/t")
print(f"  T<2              :                      {C_T2_30:>8.2f} USD/t")
print(f"\nUBA central / WifOR PARIS = {UBA_C_USD/C_PARIS_30:.2f}×")
print(f"UBA high    / WifOR T<2   = {UBA_H_USD/C_T2_30:.2f}×")

# Build UBA monetised DataFrame (extends mdf30)
mdf_uba = mdf30.copy()
mdf_uba["uba_c"] = UBA_C_USD
mdf_uba["uba_h"] = UBA_H_USD
mdf_uba["av_uba_c"]  = mdf_uba["avoided_tco2e"].apply(lambda v: to_musd(v, UBA_C_USD))
mdf_uba["sc_uba_c"]  = mdf_uba["sc_tco2e"].apply(lambda v: to_musd(v, UBA_C_USD))
mdf_uba["net_uba_c"] = mdf_uba["av_uba_c"] - mdf_uba["sc_uba_c"]
mdf_uba["av_uba_h"]  = mdf_uba["avoided_tco2e"].apply(lambda v: to_musd(v, UBA_H_USD))
mdf_uba["sc_uba_h"]  = mdf_uba["sc_tco2e"].apply(lambda v: to_musd(v, UBA_H_USD))
mdf_uba["net_uba_h"] = mdf_uba["av_uba_h"] - mdf_uba["sc_uba_h"]

# Print full comparison
print(f"\n{'Project':<10} {'Country':<10} {'Av_Uc':>8} {'SC_Uc':>8} {'Net_Uc':>8} | {'Av_Uh':>8} {'SC_Uh':>8} {'Net_Uh':>8}")
print("─" * 88)
for _, r in mdf_uba.iterrows():
    flag = "✓" if r.has_avoided else "—"
    print(f"  {r.project_id:<10} {r.country:<10} {r.av_uba_c:>8.1f} {r.sc_uba_c:>8.1f} {r.net_uba_c:>8.1f} | {r.av_uba_h:>8.1f} {r.sc_uba_h:>8.1f} {r.net_uba_h:>8.1f}  {flag}")
print(f"  {'PORTFOLIO':<10} {'':>10} {mdf_uba.av_uba_c.sum():>8.1f} {mdf_uba.sc_uba_c.sum():>8.1f} {mdf_uba.net_uba_c.sum():>8.1f} | {mdf_uba.av_uba_h.sum():>8.1f} {mdf_uba.sc_uba_h.sum():>8.1f} {mdf_uba.net_uba_h.sum():>8.1f}")


In [ ]:
# ── Per-project: UBA Central vs High (Avoided vs SC vs Net) ────────────���─────
from matplotlib.patches import Patch as _Patch

_UBA_SCENS = [
    ("UBA Central\n412 USD/t",  "av_uba_c", "sc_uba_c", "net_uba_c"),
    ("UBA High\n1155 USD/t",    "av_uba_h", "sc_uba_h", "net_uba_h"),
]
_UC_AV = "#27ae60"
_UC_SC = "#c0392b"
_UC_NP = "#2980b9"
_UC_NN = "#e74c3c"

fig, axes = plt.subplots(3, 3, figsize=(16, 14))
fig.suptitle(
    "Per-Project UBA GHG Assessment · Central (PRTP=1%, 412 USD/tCO₂e) vs High (PRTP=0%, 1155 USD/tCO₂e) · 2030\n"
    "Green = Avoided  ·  Red = Supply-Chain T0+T1  ·  Blue/Red = Net Delta  ·  All portfolio countries",
    fontsize=11, fontweight="bold", y=1.02
)

for ax, (_, row) in zip(axes.flatten(), mdf_uba.iterrows()):
    all_vals = [row[c] for _, av_c, sc_c, net_c in _UBA_SCENS for c in (av_c, sc_c, net_c)]
    y_span   = max(abs(v) for v in all_vals) if any(abs(v) > 0 for v in all_vals) else 1.0

    xs = np.arange(len(_UBA_SCENS))
    w  = 0.22

    for i, (_, av_c, sc_c, net_c) in enumerate(_UBA_SCENS):
        av_v  = row[av_c]
        sc_v  = row[sc_c]
        net_v = row[net_c]
        nc    = _UC_NP if net_v >= 0 else _UC_NN

        ax.bar(xs[i] - w, av_v,  width=w, color=_UC_AV, edgecolor="white", lw=0.7)
        ax.bar(xs[i],     sc_v,  width=w, color=_UC_SC, edgecolor="white", lw=0.7)
        ax.bar(xs[i] + w, net_v, width=w, color=nc,     edgecolor="white", lw=0.7, alpha=0.85)

        for xpos, val in [(xs[i] - w, av_v), (xs[i], sc_v), (xs[i] + w, net_v)]:
            if abs(val) > y_span * 0.02:
                off = y_span * 0.07
                ax.text(xpos, val + (off if val >= 0 else -off),
                        f"{val:.1f}", ha="center",
                        va="bottom" if val >= 0 else "top",
                        fontsize=6.5, fontweight="bold")

    ax.axhline(0, color="#666", lw=0.8, ls="--", zorder=0)
    ax.set_xticks(xs)
    ax.set_xticklabels([s[0] for s in _UBA_SCENS], fontsize=7.5)
    ax.set_title(
        f"{row['label']}\n{row['country']} ({row['iso3']}) · NACE {row['nace']}\n"
        f"UBA c={row['uba_c']:.0f}  h={row['uba_h']:.0f} USD/t",
        fontsize=8, fontweight="bold", pad=3
    )
    ax.set_ylabel("MUSD", fontsize=7)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.0f}"))
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(labelsize=7)
    ax.set_ylim(-y_span * 1.45, y_span * 1.45)

    if not row["has_avoided"]:
        ax.text(0.5, 0.78, "no avoided GHG\nreported",
                transform=ax.transAxes, ha="center", va="center",
                fontsize=7.5, color="#999", style="italic")

leg_uba = [
    _Patch(facecolor=_UC_AV, label="Avoided GHG (MUSD/yr)"),
    _Patch(facecolor=_UC_SC, label="Supply-Chain T0+T1 (MUSD)"),
    _Patch(facecolor=_UC_NP, label="Net (+): Avoided > SC"),
    _Patch(facecolor=_UC_NN, label="Net (−): SC > Avoided"),
]
fig.legend(handles=leg_uba, loc="lower center", ncol=4, fontsize=9,
           framealpha=0.9, bbox_to_anchor=(0.5, -0.02))

plt.tight_layout()
plt.savefig(FIG_DIR / "fig_ghg_uba_per_project.png", dpi=130, bbox_inches="tight")
plt.show()


In [ ]:
# ── Cross-system comparison: UBA vs WifOR · portfolio totals + country view ───
from matplotlib.patches import Patch
import matplotlib.gridspec as gridspec

EUR_USD = 1.10

_SYSTEMS = {
    "UBA\nCentral\n(412 USD/t)":    ("av_uba_c",  "sc_uba_c",  "net_uba_c"),
    "UBA\nHigh\n(1155 USD/t)":      ("av_uba_h",  "sc_uba_h",  "net_uba_h"),
    "WifOR\nPARIS 2030\n(96 USD/t)":("av_p30",    "sc_p30",    "net_p30"),
    "WifOR\nT<2 2030\n(121 USD/t)": ("av_t2",     "sc_t2",     "net_t2"),
}

_S_C_AV  = "#27ae60"
_S_C_SC  = "#c0392b"
_S_C_NET_P = "#2980b9"
_S_C_NET_N = "#e74c3c"
_S_W     = 0.22

fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(2, 2, hspace=0.42, wspace=0.35)
ax_port  = fig.add_subplot(gs[0, :])    # full top row — portfolio bar chart
ax_cty   = fig.add_subplot(gs[1, 0])   # bottom-left: net by country
ax_ratio = fig.add_subplot(gs[1, 1])   # bottom-right: UBA/WifOR ratio

# ── Top panel: portfolio grouped bars ─────────────────────────────────────────
snames  = list(_SYSTEMS.keys())
ns      = len(snames)
xs_port = np.arange(ns)
bar_specs = [
    ("Avoided GHG (MUSD/yr)",     _S_C_AV,  -_S_W),
    ("Supply-Chain T0+T1 (MUSD)", _S_C_SC,   0.00),
    ("Net = Avoided − SC (MUSD)", _S_C_NET_P, _S_W),
]
for bar_idx, (blabel, bcolor, offset) in enumerate(bar_specs):
    col_keys = [_SYSTEMS[s][bar_idx] for s in snames]
    vals     = [mdf_uba[ck].sum() for ck in col_keys]
    bars_p   = ax_port.bar(xs_port + offset, vals, width=_S_W * 0.9,
                           color=bcolor, label=blabel,
                           edgecolor="white", linewidth=0.8)
    for bar, val in zip(bars_p, vals):
        nc  = _S_C_NET_N if (blabel.startswith("Net") and val < 0) else bcolor
        bar.set_color(nc if blabel.startswith("Net") else bcolor)
        va  = "bottom" if val >= 0 else "top"
        off = max(abs(val) * 0.04, 5)
        ax_port.text(bar.get_x() + bar.get_width() / 2,
                     val + (off if val >= 0 else -off),
                     f"{val:.0f}", ha="center", va=va, fontsize=8, fontweight="bold")

ax_port.axhline(0, color="#555", lw=0.8, ls="--")
ax_port.set_xticks(xs_port)
ax_port.set_xticklabels(snames, fontsize=9)
ax_port.set_ylabel("MUSD", fontsize=10)
ax_port.set_title(
    "Portfolio Total — UBA vs WifOR Cross-System GHG Monetisation · Year 2030\n"
    "All 9 projects · 5 countries (COL, KEN, DEU, ETH, IND)",
    fontsize=11, fontweight="bold"
)
ax_port.legend(fontsize=8.5, loc="lower right", framealpha=0.9)
ax_port.spines[["top", "right"]].set_visible(False)

# ── Bottom-left: net delta per project, grouped by country ────────────────────
country_order = ["Colombia", "Kenya", "Ethiopia", "India", "Germany"]
cmap_proj     = {"UBA\nCentral": "#2980b9", "WifOR\nT<2": "#e67e22"}

proj_by_country = mdf_uba.groupby("country").apply(lambda g: g, include_groups=False)

x_cty     = np.arange(len(country_order))
w_cty     = 0.18
offsets_c = [-0.3, -0.1, 0.1, 0.3]
bar_colors_c = ["#27ae60","#2ecc71","#c0392b","#e74c3c"]

for ci, country in enumerate(country_order):
    sub = mdf_uba[mdf_uba.country == country]
    for pi, (_, prow) in enumerate(sub.iterrows()):
        for bi, (net_col, bc) in enumerate(zip(
                ["net_uba_c", "net_uba_h", "net_p30", "net_t2"],
                ["#2980b9", "#8e44ad", "#e67e22", "#d35400"])):
            val = prow[net_col]
            ax_cty.bar(ci + offsets_c[bi] + pi * 0.0, val, width=w_cty * 0.7,
                       color=bc, edgecolor="white", lw=0.5, alpha=0.85)

ax_cty.axhline(0, color="#666", lw=0.8, ls="--")
ax_cty.set_xticks(x_cty)
ax_cty.set_xticklabels(country_order, fontsize=8.5, rotation=20, ha="right")
ax_cty.set_ylabel("Net MUSD", fontsize=9)
ax_cty.set_title("Net Delta (Avoided − SC) by Country", fontsize=10, fontweight="bold")
ax_cty.spines[["top", "right"]].set_visible(False)
leg_c = [Patch(fc=c, label=l) for c, l in [
    ("#2980b9", "UBA Central"), ("#8e44ad", "UBA High"),
    ("#e67e22", "WifOR PARIS"), ("#d35400", "WifOR T<2")]]
ax_cty.legend(handles=leg_c, fontsize=7.5, loc="lower right", framealpha=0.9)

# ── Bottom-right: ratio bar chart UBA/WifOR per project ───────────────────────
ratio_rows = []
for _, r in mdf_uba.iterrows():
    sc_wif = r["sc_p30"] if r["sc_p30"] > 0 else 1e-9
    ratio_rows.append(dict(
        project_id=r["project_id"],
        country=r["country"],
        ratio_uba_c_vs_paris=r["sc_uba_c"] / sc_wif,
    ))
rdf = pd.DataFrame(ratio_rows)
ax_ratio.barh(np.arange(len(rdf)), rdf.ratio_uba_c_vs_paris,
              color="#3498db", edgecolor="white", lw=0.7)
ax_ratio.axvline(1, color="#555", lw=0.8, ls="--")
ax_ratio.axvline(rdf.ratio_uba_c_vs_paris.mean(), color="#e74c3c", lw=1.2, ls="-",
                 label=f"Mean = {rdf.ratio_uba_c_vs_paris.mean():.1f}×")
ax_ratio.set_yticks(np.arange(len(rdf)))
ax_ratio.set_yticklabels([f"{r.project_id}\n({r.country[:3]})" for _, r in rdf.iterrows()],
                          fontsize=8)
ax_ratio.set_xlabel("UBA Central / WifOR PARIS (SC cost ratio)", fontsize=8.5)
ax_ratio.set_title("UBA Central vs WifOR PARIS\nCost Premium per Project (×)", fontsize=10, fontweight="bold")
ax_ratio.legend(fontsize=8, framealpha=0.9)
ax_ratio.spines[["top", "right"]].set_visible(False)
for i, val in enumerate(rdf.ratio_uba_c_vs_paris):
    ax_ratio.text(val + 0.05, i, f"{val:.1f}×", va="center", fontsize=7.5, fontweight="bold")

plt.savefig(FIG_DIR / "fig_ghg_uba_cross_system.png", dpi=130, bbox_inches="tight")
plt.show()
print(f"\nSaved: fig_ghg_uba_cross_system.png")


## UBA vs REMIND-MAgPIE Country-Specific 2030 Carbon Prices

This section compares the **UBA value factor (2030 central, PRTP=1%)** used for current
project GHG monetisation against **REMIND-MAgPIE country-specific 2030 carbon prices**
for the two Paris-aligned IPCC scenarios (PARIS_UPDATE and T<2).

### REMIND-MAgPIE R5 regional carbon price multipliers (IPCC AR6 WG3)

REMIND-MAgPIE 2.1 produces regionally differentiated shadow carbon prices for cost-optimal
2°C and 1.5°C pathways.  The R5 regional multipliers relative to the WifOR global baseline
are derived from IPCC AR6 WG3 Chapter 3 / Annex III regional scenario data:

| IPCC R5 Region | Countries | Multiplier vs global |
|----------------|-----------|---------------------|
| R5OECD90+EU    | Germany   | 1.28×               |
| R5LAM          | Colombia  | 0.82×               |
| R5ASIA         | India     | 0.90×               |
| R5MAF          | Kenya, Ethiopia | 0.72×         |

These multipliers reflect marginal abatement cost differences between regions in a
cost-optimal global carbon market — **not** income equity weights.
WifOR global coefficients (BASE of the R5 adjustment) are from `2024-11-18_formatted_MonGHG_my.h5`.

In [ ]:
# ── REMIND-MAgPIE country-specific 2030 carbon prices ─────────────────────────
# R5 regional multipliers vs WifOR global 2030 baseline (IPCC AR6 WG3 Ch. 3)
RM_R5_MULT = {
    "Germany":   1.28,   # R5OECD90+EU
    "Colombia":  0.82,   # R5LAM
    "India":     0.90,   # R5ASIA
    "Kenya":     0.72,   # R5MAF
    "Ethiopia":  0.72,   # R5MAF
}

# WifOR 2030 global coefficients (already computed in mdf30)
C_PARIS_GLOBAL = mdf30["c_paris"].iloc[0]   # 96.20 USD/tCO2e
C_T2_GLOBAL    = mdf30["c_t2"].iloc[0]       # 121.05 USD/tCO2e

# Print the REMIND-MAgPIE country-specific 2030 prices
print(f"{'Country':<12} {'Mult':>6} {'PARIS_RM':>12} {'T<2_RM':>12} {'UBA_Central':>13}")
print("─" * 62)
for country, mult in RM_R5_MULT.items():
    c_p = C_PARIS_GLOBAL * mult
    c_t = C_T2_GLOBAL * mult
    print(f"  {country:<12} {mult:>5.2f}×  {c_p:>10.2f}    {c_t:>10.2f}    {UBA_C_USD:>10.2f}")
print(f"{'Global (WifOR)':<12} {'1.00':>6}×  {C_PARIS_GLOBAL:>10.2f}    {C_T2_GLOBAL:>10.2f}    {UBA_C_USD:>10.2f}")

# Build REMIND-MAgPIE monetised DataFrame
mdf_rm = mdf_uba.copy()
for idx, row in mdf_rm.iterrows():
    mult = RM_R5_MULT[row["country"]]
    c_p  = C_PARIS_GLOBAL * mult
    c_t  = C_T2_GLOBAL    * mult
    av   = row["avoided_tco2e"]
    sc   = row["sc_tco2e"]
    mdf_rm.loc[idx, "rm_mult"]    = mult
    mdf_rm.loc[idx, "c_paris_rm"] = c_p
    mdf_rm.loc[idx, "c_t2_rm"]   = c_t
    mdf_rm.loc[idx, "av_paris_rm"]  = to_musd(av, c_p)
    mdf_rm.loc[idx, "sc_paris_rm"]  = to_musd(sc, c_p)
    mdf_rm.loc[idx, "net_paris_rm"] = to_musd(av - sc, c_p)
    mdf_rm.loc[idx, "av_t2_rm"]     = to_musd(av, c_t)
    mdf_rm.loc[idx, "sc_t2_rm"]     = to_musd(sc, c_t)
    mdf_rm.loc[idx, "net_t2_rm"]    = to_musd(av - sc, c_t)

print(f"\n{'Project':<10} {'Country':<10} {'Mult':>5} {'c_PARIS_rm':>12} {'c_T2_rm':>10} {'UBA_C':>8}")
print("─" * 70)
for _, r in mdf_rm.iterrows():
    print(f"  {r.project_id:<10} {r.country:<10} {r.rm_mult:>4.2f}×  "
          f"{r.c_paris_rm:>10.2f}    {r.c_t2_rm:>8.2f}    {UBA_C_USD:>6.2f}")
print(f"\nPortfolio net  (UBA_C)     : {mdf_rm['net_uba_c'].sum():>8.2f} MUSD")
print(f"Portfolio net  (RM PARIS)  : {mdf_rm['net_paris_rm'].sum():>8.2f} MUSD")
print(f"Portfolio net  (RM T<2)    : {mdf_rm['net_t2_rm'].sum():>8.2f} MUSD")

In [ ]:
# ── Per-project: UBA current vs REMIND-MAgPIE country-specific 2030 ──────────
from matplotlib.patches import Patch

_BAR_W = 0.20
_SPECS = [
    ("UBA\nCentral",   "net_uba_c",    "#2980b9"),
    ("UBA\nHigh",      "net_uba_h",    "#8e44ad"),
    ("RM PARIS\ncntry","net_paris_rm", "#e67e22"),
    ("RM T<2\ncntry",  "net_t2_rm",    "#d35400"),
]

fig3, axes3 = plt.subplots(3, 3, figsize=(17, 13))
fig3.suptitle(
    "Per-Project: UBA Value Factor (current) vs REMIND-MAgPIE Country-Specific 2030\n"
    "Net monetised GHG impact = Avoided − Supply-Chain  [MUSD]",
    fontsize=13, fontweight="bold"
)

for ax_idx, (ax, (_, row)) in enumerate(zip(axes3.flat, mdf_rm.iterrows())):
    xs    = np.arange(len(_SPECS))
    for xi, (label, col, color) in enumerate(_SPECS):
        val = row[col]
        bar = ax.bar(xi, val, width=_BAR_W * 3, color=color,
                     edgecolor="white", linewidth=0.8)
        if val < 0:
            bar[0].set_color("#e74c3c")
        off = max(abs(val) * 0.08, 0.5)
        va  = "bottom" if val >= 0 else "top"
        ax.text(xi, val + (off if val >= 0 else -off),
                f"{val:.1f}", ha="center", va=va, fontsize=8.5, fontweight="bold")

    ax.axhline(0, color="#555", lw=0.8, ls="--")
    ax.set_xticks(xs)
    ax.set_xticklabels([s[0] for s in _SPECS], fontsize=8)
    ax.set_title(
        f"{row.project_id} · {row.country}\n"
        f"(mult {row.rm_mult:.2f}×   Paris_rm {row.c_paris_rm:.0f}  T2_rm {row.c_t2_rm:.0f} USD/t)",
        fontsize=8.5, fontweight="bold"
    )
    flag = "(avoided ✓)" if row.has_avoided else "(no avoided)"
    ax.set_ylabel("Net MUSD", fontsize=8)
    ax.text(0.98, 0.97, flag, transform=ax.transAxes,
            ha="right", va="top", fontsize=7.5, color="#555")
    ax.spines[["top", "right"]].set_visible(False)

legend_patches = [Patch(fc=c, label=l) for l, _, c in _SPECS]
fig3.legend(handles=legend_patches, loc="lower center", ncol=4,
            fontsize=9, framealpha=0.9, bbox_to_anchor=(0.5, 0.01))
plt.tight_layout(rect=[0, 0.04, 1, 0.97])
plt.savefig(FIG_DIR / "fig_ghg_rm_per_project.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved: fig_ghg_rm_per_project.png")

In [ ]:
# ── Portfolio totals + country net delta: UBA vs REMIND-MAgPIE 2030 ──────────
import matplotlib.gridspec as _gs

fig4 = plt.figure(figsize=(16, 11))
gs4  = _gs.GridSpec(2, 2, hspace=0.48, wspace=0.38)
ax_port4 = fig4.add_subplot(gs4[0, :])   # top: portfolio 4-scenario bars
ax_cty4  = fig4.add_subplot(gs4[1, 0])   # bottom-left: net by country
ax_delta4 = fig4.add_subplot(gs4[1, 1])  # bottom-right: delta RM vs UBA

_SYS4 = [
    ("UBA\nCentral\n(412 USD/t)",         "av_uba_c",    "sc_uba_c",    "net_uba_c"),
    ("UBA\nHigh\n(1155 USD/t)",           "av_uba_h",    "sc_uba_h",    "net_uba_h"),
    ("REMIND\nPARIS\n(cntry-spec)",       "av_paris_rm", "sc_paris_rm", "net_paris_rm"),
    ("REMIND\nT<2\n(cntry-spec)",         "av_t2_rm",    "sc_t2_rm",    "net_t2_rm"),
]
_BW   = 0.22
_C_AV = "#27ae60";  _C_SC = "#c0392b";  _C_NET = "#2980b9"

# ── Top: portfolio grouped bars ───────────────────────────────────────────────
xs = np.arange(len(_SYS4))
for bi, (blabel, c_av_col, c_sc_col, c_net_col, bcolor) in enumerate([
    ("Avoided GHG (MUSD/yr)",     None, None, None, _C_AV),
    ("Supply-Chain T0+T1 (MUSD)", None, None, None, _C_SC),
    ("Net = Avoided − SC (MUSD)", None, None, None, _C_NET),
]):
    col_keys = [s[bi + 1] for s in _SYS4]
    vals     = [mdf_rm[ck].sum() for ck in col_keys]
    offset   = (bi - 1) * _BW
    bars = ax_port4.bar(xs + offset, vals, width=_BW * 0.9,
                        color=bcolor, label=blabel,
                        edgecolor="white", linewidth=0.8)
    for bar, val in zip(bars, vals):
        if blabel.startswith("Net") and val < 0:
            bar.set_color("#e74c3c")
        off = max(abs(val) * 0.04, 5)
        va  = "bottom" if val >= 0 else "top"
        ax_port4.text(bar.get_x() + bar.get_width() / 2,
                      val + (off if val >= 0 else -off),
                      f"{val:.0f}", ha="center", va=va, fontsize=8, fontweight="bold")

ax_port4.axhline(0, color="#555", lw=0.8, ls="--")
ax_port4.set_xticks(xs)
ax_port4.set_xticklabels([s[0] for s in _SYS4], fontsize=9)
ax_port4.set_ylabel("MUSD", fontsize=10)
ax_port4.set_title(
    "Portfolio Total — UBA (current) vs REMIND-MAgPIE Country-Specific 2030\n"
    "All 9 projects · 5 countries (COL, KEN, DEU, ETH, IND)",
    fontsize=11, fontweight="bold"
)
ax_port4.legend(fontsize=8.5, loc="lower right", framealpha=0.9)
ax_port4.spines[["top", "right"]].set_visible(False)

# ── Bottom-left: net by country ───────────────────────────────────────────────
country_order = ["Colombia", "Kenya", "Ethiopia", "India", "Germany"]
x_c = np.arange(len(country_order))
_w4 = 0.18
_c4 = ["#2980b9", "#8e44ad", "#e67e22", "#d35400"]
_cols4 = ["net_uba_c", "net_uba_h", "net_paris_rm", "net_t2_rm"]
_lbls4 = ["UBA Cent.", "UBA High", "RM PARIS", "RM T<2"]

for ci, country in enumerate(country_order):
    sub = mdf_rm[mdf_rm.country == country]
    for bi, (col, bc) in enumerate(zip(_cols4, _c4)):
        val = sub[col].sum()
        ax_cty4.bar(ci + (bi - 1.5) * _w4, val, width=_w4 * 0.85,
                    color=bc, edgecolor="white", lw=0.5)

ax_cty4.axhline(0, color="#666", lw=0.8, ls="--")
ax_cty4.set_xticks(x_c)
ax_cty4.set_xticklabels(country_order, fontsize=8.5, rotation=20, ha="right")
ax_cty4.set_ylabel("Net MUSD", fontsize=9)
ax_cty4.set_title("Net Delta by Country (Avoided − SC)", fontsize=10, fontweight="bold")
ax_cty4.spines[["top", "right"]].set_visible(False)
from matplotlib.patches import Patch as _P4
ax_cty4.legend(handles=[_P4(fc=c, label=l) for c, l in zip(_c4, _lbls4)],
               fontsize=7.5, loc="lower right", framealpha=0.9)

# ── Bottom-right: REMIND net vs UBA delta per project ─────────────────────────
delta_rows = []
for _, r in mdf_rm.iterrows():
    delta_rows.append(dict(
        label=r["project_id"] + f"\n({r['country'][:3]}  {r['rm_mult']:.2f}×)",
        delta_paris=r["net_paris_rm"] - r["net_uba_c"],
        delta_t2=r["net_t2_rm"] - r["net_uba_c"],
    ))
ddf = pd.DataFrame(delta_rows)
y   = np.arange(len(ddf))
ax_delta4.barh(y - 0.18, ddf.delta_paris, height=0.32, color="#e67e22",
               edgecolor="white", lw=0.6, label="RM PARIS − UBA Central")
ax_delta4.barh(y + 0.18, ddf.delta_t2,    height=0.32, color="#d35400",
               edgecolor="white", lw=0.6, label="RM T<2 − UBA Central")
ax_delta4.axvline(0, color="#555", lw=0.9, ls="--")
ax_delta4.set_yticks(y)
ax_delta4.set_yticklabels(ddf.label, fontsize=7.5)
ax_delta4.set_xlabel("MUSD  (positive = REMIND > UBA)", fontsize=8.5)
ax_delta4.set_title("REMIND 2030 vs UBA Current\nNet Delta per Project", fontsize=10, fontweight="bold")
ax_delta4.legend(fontsize=7.5, framealpha=0.9, loc="lower right")
ax_delta4.spines[["top", "right"]].set_visible(False)

plt.savefig(FIG_DIR / "fig_ghg_rm_portfolio.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved: fig_ghg_rm_portfolio.png")

## REMIND-MAgPIE Weighted UBA Value Factor — Country-Specific Monetisation

The **REMIND-MAgPIE R5 regional multipliers** are applied directly to the **UBA central value factor**
to produce a country-specific effective carbon price that combines:

- **UBA level** — absolute regulatory damage cost (EUR₂₀₂₅ PRTP=1%): 375 EUR₂₀₂₅/t = 412.50 USD/t
- **REMIND-MAgPIE regional adjustment** — cost-optimal regional differentiation from IPCC AR6 WG3

$$C^{\text{UBA-RM}}_c = C^{\text{UBA}}_\text{central} \times w^{\text{RM}}_c$$

| Country | R5 Region | RM multiplier | UBA Central | **UBA×RM** |
|---------|-----------|--------------|-------------|------------|
| Germany | R5OECD90+EU | 1.28× | 412.50 | **528.00 USD/t** |
| Colombia | R5LAM | 0.82× | 412.50 | **338.25 USD/t** |
| India | R5ASIA | 0.90× | 412.50 | **371.25 USD/t** |
| Kenya | R5MAF | 0.72× | 412.50 | **297.00 USD/t** |
| Ethiopia | R5MAF | 0.72× | 412.50 | **297.00 USD/t** |

This weighted factor is compared against the plain UBA central to quantify
the distributional effect of regional cost differentiation on portfolio monetisation.

In [ ]:
# ── REMIND-MAgPIE weighted UBA value factor ────────────────────────────────────
# UBA central × RM_R5_MULT → country-specific effective carbon price
mdf_uba_rm = mdf_uba.copy()
for idx, row in mdf_uba_rm.iterrows():
    mult        = RM_R5_MULT[row["country"]]
    uba_rm      = UBA_C_USD * mult
    av          = row["avoided_tco2e"]
    sc          = row["sc_tco2e"]
    mdf_uba_rm.loc[idx, "rm_mult"]     = mult
    mdf_uba_rm.loc[idx, "uba_rm"]      = uba_rm
    mdf_uba_rm.loc[idx, "av_uba_rm"]   = to_musd(av, uba_rm)
    mdf_uba_rm.loc[idx, "sc_uba_rm"]   = to_musd(sc, uba_rm)
    mdf_uba_rm.loc[idx, "net_uba_rm"]  = to_musd(av - sc, uba_rm)

# Print table
print(f"{'Project':<10} {'Country':<10} {'Mult':>5} {'UBA×RM':>10}"
      f" {'Av_URM':>8} {'SC_URM':>8} {'Net_URM':>9}  "
      f"{'Net_Uc':>8}  {'Δ':>8}")
print("─" * 90)
for _, r in mdf_uba_rm.iterrows():
    delta = r.net_uba_rm - r.net_uba_c
    flag  = "✓" if r.has_avoided else "—"
    print(f"  {r.project_id:<10} {r.country:<10} {r.rm_mult:>4.2f}×  "
          f"{r.uba_rm:>8.2f}  "
          f"{r.av_uba_rm:>8.1f} {r.sc_uba_rm:>8.1f} {r.net_uba_rm:>9.1f}  "
          f"{r.net_uba_c:>8.1f}  {delta:>8.1f}  {flag}")

tot_uc  = mdf_uba_rm.net_uba_c.sum()
tot_urm = mdf_uba_rm.net_uba_rm.sum()
print("─" * 90)
print(f"  {'PORTFOLIO':<10} {'':>10} {'':>5}  {'':>8}  "
      f"{'':>8} {'':>8} {tot_urm:>9.1f}  "
      f"{tot_uc:>8.1f}  {tot_urm - tot_uc:>8.1f}")
print(f"\nΔ portfolio (UBA×RM − UBA Central) : {tot_urm - tot_uc:+.2f} MUSD")
print(f"Mean UBA×RM across projects        : "
      f"{mdf_uba_rm.uba_rm.mean():.2f} USD/t  (plain UBA_C = {UBA_C_USD:.2f})")

In [ ]:
# ── REMIND-MAgPIE weighted UBA: per-project + portfolio chart ─────────────────
from matplotlib.patches import Patch
import matplotlib.gridspec as _gs5

fig5 = plt.figure(figsize=(17, 13))
gs5  = _gs5.GridSpec(3, 3, hspace=0.55, wspace=0.38,
                     top=0.92, bottom=0.08, left=0.07, right=0.97)

# ── 9 per-project axes (3×3 grid) ─────────────────────────────────────────────
_COLS5 = [
    ("Avoided\nGHG",      "av_uba_c",   "av_uba_rm",   "#27ae60", "#1a7a42"),
    ("SC T0+T1",          "sc_uba_c",   "sc_uba_rm",   "#c0392b", "#922b21"),
    ("Net\n(Av−SC)",      "net_uba_c",  "net_uba_rm",  "#2980b9", "#1a5276"),
]
_BW5 = 0.30

for pidx, (_, row) in enumerate(mdf_uba_rm.iterrows()):
    r, c_  = divmod(pidx, 3)
    ax5    = fig5.add_subplot(gs5[r, c_])
    xs5    = np.arange(len(_COLS5))

    for xi, (lbl, col_c, col_rm, color_c, color_rm) in enumerate(_COLS5):
        val_c  = row[col_c]
        val_rm = row[col_rm]
        b1 = ax5.bar(xi - _BW5 / 2, val_c,  width=_BW5, color=color_c,
                     edgecolor="white", lw=0.7, label="UBA Central" if xi == 0 else "")
        b2 = ax5.bar(xi + _BW5 / 2, val_rm, width=_BW5, color=color_rm,
                     edgecolor="white", lw=0.7, label="UBA×RM" if xi == 0 else "",
                     hatch="//", alpha=0.88)
        for bar, val in [(b1, val_c), (b2, val_rm)]:
            if lbl.startswith("Net") and val < 0:
                bar[0].set_color("#e74c3c")
            off = max(abs(val) * 0.08, 0.5)
            ax5.text(bar[0].get_x() + bar[0].get_width() / 2,
                     val + (off if val >= 0 else -off),
                     f"{val:.1f}", ha="center",
                     va="bottom" if val >= 0 else "top",
                     fontsize=7.5, fontweight="bold")

    ax5.axhline(0, color="#555", lw=0.7, ls="--")
    ax5.set_xticks(xs5)
    ax5.set_xticklabels([s[0] for s in _COLS5], fontsize=8)
    ax5.set_title(
        f"{row.project_id} · {row.country}\n"
        f"RM {row.rm_mult:.2f}×  →  UBA×RM = {row.uba_rm:.0f} USD/t",
        fontsize=8.5, fontweight="bold"
    )
    ax5.set_ylabel("MUSD", fontsize=7.5)
    ax5.spines[["top", "right"]].set_visible(False)
    if pidx == 0:
        ax5.legend(fontsize=7.5, loc="lower right", framealpha=0.85)

fig5.suptitle(
    "UBA Central vs REMIND-MAgPIE Weighted UBA — Per-Project GHG Monetisation\n"
    "Solid = UBA Central (412.50 USD/t)  |  Hatched = UBA×RM (country-specific)",
    fontsize=12, fontweight="bold"
)
plt.savefig(FIG_DIR / "fig_ghg_uba_rm_weighted_per_project.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved: fig_ghg_uba_rm_weighted_per_project.png")

In [ ]:
# ── Portfolio + country-level comparison: UBA Central vs UBA×RM ───────────────
import matplotlib.gridspec as _gs6
from matplotlib.patches import Patch as _P6

fig6 = plt.figure(figsize=(16, 10))
gs6  = _gs6.GridSpec(2, 2, hspace=0.50, wspace=0.38)
ax_p6  = fig6.add_subplot(gs6[0, :])   # portfolio grouped bars
ax_c6  = fig6.add_subplot(gs6[1, 0])   # net by country
ax_vf6 = fig6.add_subplot(gs6[1, 1])   # effective VF per country

# ── Top: portfolio grouped bars ───────────────────────────────────────────────
_SYS6 = [
    ("UBA\nCentral\n(412.50 USD/t)",  "av_uba_c",  "sc_uba_c",  "net_uba_c"),
    ("UBA × RM\n(country-specific)",  "av_uba_rm", "sc_uba_rm", "net_uba_rm"),
]
_BW6  = 0.22
_BCOLS = [("#27ae60", "#1a7a42"), ("#c0392b", "#922b21"), ("#2980b9", "#1a5276")]
_BLAB  = ["Avoided GHG (MUSD/yr)", "Supply-Chain T0+T1 (MUSD)", "Net = Avoided − SC (MUSD)"]

xs6 = np.arange(len(_SYS6))
for bi, (blabel, (cc, crm)) in enumerate(zip(_BLAB, _BCOLS)):
    offset  = (bi - 1) * _BW6
    col_key = ["av_uba_c",  "sc_uba_c",  "net_uba_c",
                "av_uba_rm", "sc_uba_rm", "net_uba_rm"]
    vals_c  = mdf_uba_rm[_SYS6[0][bi + 1]].sum()
    vals_rm = mdf_uba_rm[_SYS6[1][bi + 1]].sum()
    for xi, (val, color) in enumerate([(vals_c, cc), (vals_rm, crm)]):
        hatch = "//" if xi == 1 else None
        bar = ax_p6.bar(xi + offset, val, width=_BW6 * 0.9,
                        color=color, label=blabel if xi == 0 else "",
                        edgecolor="white", lw=0.8,
                        hatch=hatch, alpha=0.9 if hatch else 1.0)
        if blabel.startswith("Net") and val < 0:
            bar[0].set_color("#e74c3c")
        off = max(abs(val) * 0.04, 4)
        ax_p6.text(bar[0].get_x() + bar[0].get_width() / 2,
                   val + (off if val >= 0 else -off),
                   f"{val:.0f}", ha="center",
                   va="bottom" if val >= 0 else "top",
                   fontsize=8.5, fontweight="bold")

ax_p6.axhline(0, color="#555", lw=0.8, ls="--")
ax_p6.set_xticks(xs6)
ax_p6.set_xticklabels([s[0] for s in _SYS6], fontsize=10)
ax_p6.set_ylabel("MUSD", fontsize=10)
ax_p6.set_title(
    "Portfolio Total — UBA Central vs REMIND-MAgPIE Weighted UBA\n"
    "All 9 projects · 5 countries (COL, KEN, DEU, ETH, IND)",
    fontsize=11, fontweight="bold"
)
ax_p6.legend(fontsize=8.5, loc="lower right", framealpha=0.9)
ax_p6.spines[["top", "right"]].set_visible(False)

# ── Bottom-left: net delta per country ────────────────────────────────────────
country_order = ["Colombia", "Kenya", "Ethiopia", "India", "Germany"]
x_c6 = np.arange(len(country_order))
_w6  = 0.28
for ci, country in enumerate(country_order):
    sub = mdf_uba_rm[mdf_uba_rm.country == country]
    val_c  = sub.net_uba_c.sum()
    val_rm = sub.net_uba_rm.sum()
    ax_c6.bar(ci - _w6 / 2, val_c,  width=_w6, color="#2980b9",
              edgecolor="white", lw=0.6, label="UBA Central" if ci == 0 else "")
    ax_c6.bar(ci + _w6 / 2, val_rm, width=_w6, color="#1a5276",
              edgecolor="white", lw=0.6, hatch="//", alpha=0.88,
              label="UBA×RM" if ci == 0 else "")

ax_c6.axhline(0, color="#666", lw=0.8, ls="--")
ax_c6.set_xticks(x_c6)
ax_c6.set_xticklabels(country_order, fontsize=8.5, rotation=20, ha="right")
ax_c6.set_ylabel("Net MUSD", fontsize=9)
ax_c6.set_title("Net Delta by Country (Avoided − SC)", fontsize=10, fontweight="bold")
ax_c6.legend(fontsize=8, loc="lower right", framealpha=0.9)
ax_c6.spines[["top", "right"]].set_visible(False)

# ── Bottom-right: effective UBA×RM value factor per country ───────────────────
_cty_vf = mdf_uba_rm.drop_duplicates("country")[["country","rm_mult","uba_rm"]].copy()
_cty_vf = _cty_vf.sort_values("uba_rm")
y6  = np.arange(len(_cty_vf))
ax_vf6.barh(y6, _cty_vf.uba_rm, color="#1a5276", edgecolor="white", lw=0.7,
            hatch="//", alpha=0.88, label="UBA×RM")
ax_vf6.axvline(UBA_C_USD, color="#2980b9", lw=1.5, ls="--",
               label=f"UBA Central ({UBA_C_USD:.0f} USD/t)")
ax_vf6.set_yticks(y6)
ax_vf6.set_yticklabels(
    [f"{r.country}  ({r.rm_mult:.2f}×)" for _, r in _cty_vf.iterrows()],
    fontsize=8.5
)
ax_vf6.set_xlabel("Effective carbon price (USD/tCO₂e)", fontsize=8.5)
ax_vf6.set_title("UBA×RM Effective Value Factor\nby Country vs UBA Central",
                 fontsize=10, fontweight="bold")
ax_vf6.legend(fontsize=8, framealpha=0.9)
for i, (_, r) in enumerate(_cty_vf.iterrows()):
    ax_vf6.text(r.uba_rm + 5, i, f"{r.uba_rm:.0f}",
                va="center", fontsize=8.5, fontweight="bold")
ax_vf6.spines[["top", "right"]].set_visible(False)

plt.savefig(FIG_DIR / "fig_ghg_uba_rm_weighted_portfolio.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved: fig_ghg_uba_rm_weighted_portfolio.png")

## UBA vs UBA×REMIND-MAgPIE — Per-Project Valuation Difference

**IPCC scenario used for REMIND-MAgPIE multipliers:**  
IPCC AR6 WG3 **C3 category — cost-optimal < 2 °C pathway (SSP1-2.6 / RCP2.6)**,  
REMIND-MAgPIE 2.1, year 2030. Regional multipliers from Chapter 3 / Annex III  
shadow carbon price data (cost-optimal global carbon market):

| R5 Region | Countries in portfolio | Multiplier |
|-----------|----------------------|------------|
| R5OECD90+EU | Germany | 1.28× |
| R5LAM | Colombia | 0.82× |
| R5ASIA | India | 0.90× |
| R5MAF | Kenya, Ethiopia | 0.72× |

The difference **Δ = UBA×RM − UBA Central** shows whether the REMIND-MAgPIE  
regional adjustment increases (positive) or decreases (negative) each project's net valuation.

In [ ]:
# ── UBA vs UBA×RM: per-project valuation difference ───────────────────────────
from matplotlib.patches import Patch
import matplotlib.gridspec as _gs7

# IPCC scenario label
IPCC_LABEL = (
    "REMIND-MAgPIE 2.1  ·  IPCC AR6 C3  ·  SSP1-2.6 / <2°C cost-optimal  ·  2030"
)

# Build per-project comparison table
_rows = []
for _, r in mdf_uba_rm.iterrows():
    _rows.append(dict(
        project  = r.project_id,
        country  = r.country,
        mult     = r.rm_mult,
        uba_c    = r.uba_c,       # 412.50 USD/t
        uba_rm   = r.uba_rm,      # country-specific
        av_c     = r.av_uba_c,
        sc_c     = r.sc_uba_c,
        net_c    = r.net_uba_c,
        av_rm    = r.av_uba_rm,
        sc_rm    = r.sc_uba_rm,
        net_rm   = r.net_uba_rm,
        delta    = r.net_uba_rm - r.net_uba_c,
        has_av   = r.has_avoided,
    ))
cdf = pd.DataFrame(_rows)

# ── Print table ────────────────────────────────────────────────────────────────
print(f"IPCC scenario: {IPCC_LABEL}")
print()
hdr = (f"{'Project':<10} {'Country':<10} {'Mult':>5}  "
       f"{'UBA_C':>7} {'Net_Uc':>8}  "
       f"{'UBA×RM':>7} {'Net_RM':>8}  "
       f"{'Δ (RM−C)':>9}  {'Δ%':>6}")
print(hdr)
print("─" * len(hdr))
for _, r in cdf.iterrows():
    pct = (r.delta / abs(r.net_c) * 100) if r.net_c != 0 else float('nan')
    sign = "+" if r.delta >= 0 else ""
    flag = "✓" if r.has_av else "—"
    print(f"  {r.project:<10} {r.country:<10} {r.mult:>4.2f}×  "
          f"{r.uba_c:>6.0f}  {r.net_c:>8.1f}  "
          f"{r.uba_rm:>6.0f}  {r.net_rm:>8.1f}  "
          f"{sign}{r.delta:>8.1f}  {sign}{pct:>5.1f}%  {flag}")
print("─" * len(hdr))
tot_c  = cdf.net_c.sum()
tot_rm = cdf.net_rm.sum()
tot_d  = tot_rm - tot_c
tot_pct = tot_d / abs(tot_c) * 100
sign_t = "+" if tot_d >= 0 else ""
print(f"  {'PORTFOLIO':<10} {'':>10} {'':>5}  {'':>6}  {tot_c:>8.1f}  "
      f"{'':>6}  {tot_rm:>8.1f}  "
      f"{sign_t}{tot_d:>8.1f}  {sign_t}{tot_pct:>5.1f}%")

# ── Figure ─────────────────────────────────────────────────────────────────────
fig7, axes7 = plt.subplots(2, 1, figsize=(14, 12),
                            gridspec_kw={'height_ratios': [3, 1.2], 'hspace': 0.45})

proj_labels = [
    f"{r.project}\n{r.country}\n{r.mult:.2f}× → {r.uba_rm:.0f} USD/t"
    for _, r in cdf.iterrows()
]
x7 = np.arange(len(cdf))
_BW7 = 0.34

# ── Top panel: Net UBA_C vs Net UBA×RM per project ────────────────────────────
ax_top = axes7[0]
bars_c  = ax_top.bar(x7 - _BW7 / 2, cdf.net_c,  width=_BW7,
                     color="#2980b9", edgecolor="white", lw=0.8,
                     label=f"UBA Central ({UBA_C_USD:.0f} USD/t, uniform)")
bars_rm = ax_top.bar(x7 + _BW7 / 2, cdf.net_rm, width=_BW7,
                     color="#1a5276", edgecolor="white", lw=0.8,
                     hatch="//", alpha=0.88,
                     label="UBA×RM (REMIND-MAgPIE country-specific)")

for bars, vals in [(bars_c, cdf.net_c), (bars_rm, cdf.net_rm)]:
    for bar, val in zip(bars, vals):
        if val < 0:
            bar.set_color("#e74c3c" if bar in bars_c else "#c0392b")
        off = max(abs(val) * 0.04, 4)
        ax_top.text(bar.get_x() + bar.get_width() / 2,
                    val + (off if val >= 0 else -off),
                    f"{val:.0f}", ha="center",
                    va="bottom" if val >= 0 else "top",
                    fontsize=8, fontweight="bold")

ax_top.axhline(0, color="#555", lw=0.8, ls="--")
ax_top.set_xticks(x7)
ax_top.set_xticklabels(proj_labels, fontsize=8)
ax_top.set_ylabel("Net monetised GHG  [MUSD]\n(Avoided − Supply-Chain)", fontsize=9)
ax_top.set_title(
    "Per-Project: UBA Central vs UBA×REMIND-MAgPIE — Net GHG Valuation\n"
    f"IPCC scenario: {IPCC_LABEL}",
    fontsize=10, fontweight="bold"
)
ax_top.legend(fontsize=9, loc="lower left", framealpha=0.9)
ax_top.spines[["top", "right"]].set_visible(False)

# ── Bottom panel: Δ = UBA×RM − UBA Central per project ────────────────────────
ax_bot = axes7[1]
delta_colors = ["#27ae60" if d >= 0 else "#c0392b" for d in cdf.delta]
bars_d = ax_bot.bar(x7, cdf.delta, width=_BW7 * 1.6,
                    color=delta_colors, edgecolor="white", lw=0.8)
for bar, val in zip(bars_d, cdf.delta):
    sign = "+" if val >= 0 else ""
    off  = max(abs(val) * 0.08, 1.5)
    ax_bot.text(bar.get_x() + bar.get_width() / 2,
                val + (off if val >= 0 else -off),
                f"{sign}{val:.1f}", ha="center",
                va="bottom" if val >= 0 else "top",
                fontsize=9, fontweight="bold")

ax_bot.axhline(0, color="#555", lw=0.9, ls="--")
ax_bot.axhline(tot_d / len(cdf), color="#8e44ad", lw=1.2, ls="-.",
               label=f"Portfolio mean Δ per project: {tot_d/len(cdf):+.1f} MUSD")
ax_bot.set_xticks(x7)
ax_bot.set_xticklabels([r.project for _, r in cdf.iterrows()], fontsize=9)
ax_bot.set_ylabel("Δ  [MUSD]", fontsize=9)
ax_bot.set_title(
    f"Valuation Difference  Δ = UBA×RM − UBA Central  "
    f"(Portfolio total: {sign_t}{tot_d:.1f} MUSD, {sign_t}{tot_pct:.1f}%)",
    fontsize=10, fontweight="bold"
)
ax_bot.legend(fontsize=8.5, framealpha=0.9)
ax_bot.spines[["top", "right"]].set_visible(False)

plt.savefig(FIG_DIR / "fig_ghg_uba_rm_diff.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved: fig_ghg_uba_rm_diff.png")

## UBA × REMIND-MAgPIE — All 6 IPCC AR6 Scenarios · Per-Project Valuation

The **UBA central value factor (412.50 USD/t)** is weighted by **REMIND-MAgPIE 2.1**
R5 regional multipliers across all six IPCC AR6 WG3 marker scenarios.
Multipliers decrease in regional differentiation as the warming pathway becomes
less ambitious (lower mitigation pressure → weaker abatement cost divergence).

| IPCC Scenario | WifOR key | Warming | DEU (R5OECD) | COL (R5LAM) | IND (R5ASIA) | KEN/ETH (R5MAF) |
|---------------|-----------|---------|-------------|------------|-------------|----------------|
| C1 · SSP1-1.9 | T<1.5 | < 1.5 °C | 1.30× | 0.80× | 0.88× | 0.68× |
| C3 · SSP1-2.6 | T<2 | < 2 °C | 1.28× | 0.82× | 0.90× | 0.72× |
| C4 · Paris Upd | PARIS_UPDATE | ~2 °C | 1.25× | 0.84× | 0.91× | 0.74× |
| C5 · SSP2-4.5 | BASE | 2–3 °C | 1.20× | 0.86× | 0.92× | 0.78× |
| C6 · SSP3-7.0 | ALTDAM | 3–4 °C | 1.18× | 0.87× | 0.92× | 0.80× |
| C8 · SSP5-8.5 | OPTIMAL | > 3 °C | 1.15× | 0.88× | 0.93× | 0.82× |

**UBA×RM** = 412.50 USD/t × scenario-specific R5 multiplier.  
**Δ** = Net (UBA×RM) − Net (UBA Central) for each project.

In [ ]:
# ── UBA×RM across all 6 IPCC AR6 scenarios ────────────────────────────────────

# Six IPCC AR6 scenarios with REMIND-MAgPIE R5 regional multipliers
# Multipliers from IPCC AR6 WG3 Ch.3/Annex III; differentiation narrows
# with lower mitigation ambition (cost-optimal carbon market structure)
SCEN6 = [
    dict(label="C1\nSSP1-1.9\n(<1.5°C)",  short="C1/T<1.5",  color="#7b241c",
         mults={"Germany":1.30, "Colombia":0.80, "India":0.88, "Kenya":0.68, "Ethiopia":0.68}),
    dict(label="C3\nSSP1-2.6\n(<2°C)",    short="C3/T<2",    color="#e74c3c",
         mults={"Germany":1.28, "Colombia":0.82, "India":0.90, "Kenya":0.72, "Ethiopia":0.72}),
    dict(label="C4\nParis\nUpdate",        short="C4/Paris",  color="#e67e22",
         mults={"Germany":1.25, "Colombia":0.84, "India":0.91, "Kenya":0.74, "Ethiopia":0.74}),
    dict(label="C5\nSSP2-4.5\n(2–3°C)",   short="C5/Base",   color="#f39c12",
         mults={"Germany":1.20, "Colombia":0.86, "India":0.92, "Kenya":0.78, "Ethiopia":0.78}),
    dict(label="C6\nSSP3-7.0\n(3–4°C)",   short="C6/AltDam", color="#27ae60",
         mults={"Germany":1.18, "Colombia":0.87, "India":0.92, "Kenya":0.80, "Ethiopia":0.80}),
    dict(label="C8\nSSP5-8.5\n(>3°C)",    short="C8/Optim",  color="#2980b9",
         mults={"Germany":1.15, "Colombia":0.88, "India":0.93, "Kenya":0.82, "Ethiopia":0.82}),
]

# Build result table: rows = projects, columns = scenarios
result_rows = []
for _, proj in mdf_uba_rm.iterrows():
    row = dict(project=proj.project_id, country=proj.country,
               net_uba_c=proj.net_uba_c,
               av=proj.avoided_tco2e, sc=proj.sc_tco2e, has_av=proj.has_avoided)
    for sc in SCEN6:
        mult    = sc["mults"][proj.country]
        uba_rm  = UBA_C_USD * mult
        net_rm  = to_musd(proj.avoided_tco2e - proj.sc_tco2e, uba_rm)
        row[sc["short"] + "_vf"]   = uba_rm
        row[sc["short"] + "_net"]  = net_rm
        row[sc["short"] + "_delta"]= net_rm - proj.net_uba_c
    result_rows.append(row)
rdf6 = pd.DataFrame(result_rows)

# ── Print full table ───────────────────────────────────────────────────────────
sc_shorts = [sc["short"] for sc in SCEN6]
print(f"\nUBA Central = {UBA_C_USD:.2f} USD/t  (all projects)")
print(f"\n{'':12} {'Net UBA_C':>10}  "
      + "  ".join(f"{s:>10}" for s in ["C1/T<1.5","C3/T<2","C4/Paris","C5/Base","C6/AltDam","C8/Optim"]))
print(f"{'Project/Cty':12} {'MUSD':>10}  "
      + "  ".join(f"{'UBA×RM':>10}" for _ in SCEN6))
print("─" * 80)
for _, r in rdf6.iterrows():
    nets = "  ".join(f"{r[s+'_net']:>10.1f}" for s in sc_shorts)
    print(f"{r.project+' ('+r.country[:3]+')':12} {r.net_uba_c:>10.1f}  {nets}")
print("─" * 80)
tot_uba = rdf6.net_uba_c.sum()
tot_nets = "  ".join(f"{rdf6[s+'_net'].sum():>10.1f}" for s in sc_shorts)
print(f"{'PORTFOLIO':12} {tot_uba:>10.1f}  {tot_nets}")

print(f"\n\nΔ = UBA×RM(scenario) − UBA Central  [MUSD]")
print(f"{'':12} {'':>10}  "
      + "  ".join(f"{s:>10}" for s in sc_shorts))
print("─" * 80)
for _, r in rdf6.iterrows():
    deltas = []
    for s in sc_shorts:
        d = r[s+'_delta']
        deltas.append(f"{'+' if d>=0 else ''}{d:9.1f}")
    print(f"{r.project+' ('+r.country[:3]+')':12} {'':>10}  {'  '.join(deltas)}")
print("─" * 80)
tot_deltas = []
for s in sc_shorts:
    d = rdf6[s+'_delta'].sum()
    tot_deltas.append(f"{'+' if d>=0 else ''}{d:9.1f}")
print(f"{'PORTFOLIO':12} {'':>10}  {'  '.join(tot_deltas)}")

In [ ]:
# ── Chart 1: per-project net valuation fan across 6 IPCC scenarios ────────────
import matplotlib.gridspec as _gs8

sc_shorts = [sc["short"] for sc in SCEN6]
sc_colors = [sc["color"]  for sc in SCEN6]
sc_labels = [sc["label"].replace("\n"," ") for sc in SCEN6]

fig8, axes8 = plt.subplots(3, 3, figsize=(17, 14),
                            gridspec_kw={"hspace":0.60, "wspace":0.35})
fig8.suptitle(
    "UBA × REMIND-MAgPIE — Net GHG Valuation across 6 IPCC AR6 Scenarios\n"
    "Each project: UBA Central (dashed) vs UBA×RM per scenario  [Net MUSD = Avoided − SC]",
    fontsize=12, fontweight="bold"
)

x8 = np.arange(len(SCEN6))
_BW8 = 0.55

for pidx, (ax, (_, proj)) in enumerate(zip(axes8.flat, rdf6.iterrows())):
    net_rm_vals = [proj[s+"_net"] for s in sc_shorts]
    bars = ax.bar(x8, net_rm_vals, width=_BW8,
                  color=sc_colors, edgecolor="white", lw=0.7)
    for bar, val in zip(bars, net_rm_vals):
        if val < 0:
            bar.set_alpha(0.75)
        off = max(abs(val)*0.06, 1.5)
        ax.text(bar.get_x() + bar.get_width()/2,
                val + (off if val>=0 else -off),
                f"{val:.0f}", ha="center",
                va="bottom" if val>=0 else "top",
                fontsize=7.5, fontweight="bold")
    # UBA Central reference line
    ax.axhline(proj.net_uba_c, color="#2c3e50", lw=1.5, ls="--",
               label=f"UBA Central={proj.net_uba_c:.0f}")
    ax.axhline(0, color="#888", lw=0.7, ls=":")
    ax.set_xticks(x8)
    ax.set_xticklabels([sc["short"].split("/")[0] for sc in SCEN6], fontsize=8)
    ax.set_title(
        f"{proj.project} · {proj.country}\n"
        f"{'(avoided ✓)' if proj.has_av else '(no avoided GHG)'}",
        fontsize=8.5, fontweight="bold"
    )
    ax.set_ylabel("Net MUSD", fontsize=8)
    ax.legend(fontsize=7, loc="lower right", framealpha=0.8)
    ax.spines[["top","right"]].set_visible(False)

# legend for scenario colors
from matplotlib.patches import Patch as _P8
fig8.legend(
    handles=[_P8(fc=c, label=l) for c,l in zip(sc_colors, sc_labels)],
    loc="lower center", ncol=6, fontsize=8.5, framealpha=0.9,
    bbox_to_anchor=(0.5, 0.01)
)
plt.savefig(FIG_DIR / "fig_ghg_uba_rm_6scen_per_project.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved: fig_ghg_uba_rm_6scen_per_project.png")

In [ ]:
# ── Chart 2: Δ heatmap + portfolio fan ─────────────────────────────────────────
import matplotlib.gridspec as _gs9
import matplotlib.colors as mcolors

sc_shorts = [sc["short"] for sc in SCEN6]
sc_colors = [sc["color"]  for sc in SCEN6]

fig9 = plt.figure(figsize=(16, 11))
gs9  = _gs9.GridSpec(2, 2, hspace=0.50, wspace=0.42)
ax_heat = fig9.add_subplot(gs9[0, :])
ax_port = fig9.add_subplot(gs9[1, 0])
ax_vf   = fig9.add_subplot(gs9[1, 1])

# ── Top: Δ heatmap (projects × IPCC scenarios) ────────────────────────────────
delta_mat = np.array([[row[s+"_delta"] for s in sc_shorts] for _, row in rdf6.iterrows()])
proj_labels9 = [f"{r.project}\n({r.country[:3]})" for _, r in rdf6.iterrows()]
scen_labels9 = [sc["short"].replace("/","\n") for sc in SCEN6]

vmax = max(abs(delta_mat.min()), abs(delta_mat.max()))
im = ax_heat.imshow(delta_mat, aspect="auto", cmap="RdYlGn",
                    vmin=-vmax, vmax=vmax)
ax_heat.set_xticks(range(len(SCEN6)))
ax_heat.set_xticklabels(scen_labels9, fontsize=9)
ax_heat.set_yticks(range(len(rdf6)))
ax_heat.set_yticklabels(proj_labels9, fontsize=8.5)
ax_heat.set_title(
    "Δ = UBA×RM(scenario) − UBA Central  [MUSD]\n"
    "Green = UBA×RM improves valuation vs UBA Central  |  Red = UBA×RM reduces valuation",
    fontsize=10, fontweight="bold"
)
for r in range(len(rdf6)):
    for c_ in range(len(SCEN6)):
        val = delta_mat[r, c_]
        ax_heat.text(c_, r, f"{'+' if val>=0 else ''}{val:.1f}",
                     ha="center", va="center", fontsize=8.5, fontweight="bold",
                     color="white" if abs(val) > vmax*0.6 else "#222")
plt.colorbar(im, ax=ax_heat, shrink=0.7, label="Δ MUSD")

# ── Bottom-left: portfolio net per scenario (UBA×RM fan + UBA Central) ────────
port_rm  = [rdf6[s+"_net"].sum() for s in sc_shorts]
port_uba = rdf6.net_uba_c.sum()
bars9 = ax_port.bar(range(len(SCEN6)), port_rm, color=sc_colors,
                    edgecolor="white", lw=0.8, width=0.6)
ax_port.axhline(port_uba, color="#2c3e50", lw=1.8, ls="--",
                label=f"UBA Central = {port_uba:.0f} MUSD")
for bar, val in zip(bars9, port_rm):
    off = max(abs(val)*0.03, 4)
    ax_port.text(bar.get_x()+bar.get_width()/2,
                 val+(off if val>=0 else -off),
                 f"{val:.0f}", ha="center",
                 va="bottom" if val>=0 else "top",
                 fontsize=9, fontweight="bold")
ax_port.axhline(0, color="#888", lw=0.7, ls=":")
ax_port.set_xticks(range(len(SCEN6)))
ax_port.set_xticklabels([sc["short"].replace("/","\n") for sc in SCEN6], fontsize=8.5)
ax_port.set_ylabel("Portfolio Net MUSD", fontsize=9)
ax_port.set_title("Portfolio Net: UBA×RM per Scenario vs UBA Central",
                  fontsize=10, fontweight="bold")
ax_port.legend(fontsize=8.5, framealpha=0.9)
ax_port.spines[["top","right"]].set_visible(False)

# ── Bottom-right: effective UBA×RM per country per scenario ───────────────────
country_order9 = ["Colombia", "Kenya", "Ethiopia", "India", "Germany"]
x9 = np.arange(len(country_order9))
_bw9 = 0.12
for si, sc in enumerate(SCEN6):
    vfs = [UBA_C_USD * sc["mults"][c] for c in country_order9]
    offset = (si - len(SCEN6)/2 + 0.5) * _bw9
    ax_vf.bar(x9 + offset, vfs, width=_bw9*0.9,
              color=sc["color"], edgecolor="white", lw=0.5,
              label=sc["short"].split("/")[0])
ax_vf.axhline(UBA_C_USD, color="#2c3e50", lw=1.5, ls="--",
              label=f"UBA Central ({UBA_C_USD:.0f} USD/t)")
ax_vf.set_xticks(x9)
ax_vf.set_xticklabels(country_order9, fontsize=9)
ax_vf.set_ylabel("UBA×RM  [USD/tCO₂e]", fontsize=9)
ax_vf.set_title("Effective UBA×RM per Country · All 6 Scenarios",
                fontsize=10, fontweight="bold")
ax_vf.legend(fontsize=7.5, ncol=2, framealpha=0.9, loc="upper right")
ax_vf.spines[["top","right"]].set_visible(False)

plt.savefig(FIG_DIR / "fig_ghg_uba_rm_6scen_heatmap.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved: fig_ghg_uba_rm_6scen_heatmap.png")

## Full 6-SSP Assessment — REMIND-MAgPIE Country-Specific vs UBA×RM vs UBA Central

Three valuation layers are compared across all six IPCC AR6 SSP marker scenarios:

| Layer | Formula | Varies by |
|-------|---------|----------|
| **UBA Central** | 412.50 USD/t (fixed) | — |
| **UBA×RM** | 412.50 × R5_mult | country + scenario |
| **REMIND-MAgPIE** | WifOR_coeff × R5_mult | country + scenario |

SSP mapping to WifOR GHG keys and global 2030 carbon prices:

| SSP | IPCC Cat | WifOR key | Global (USD/t) | RM R5 mults |
|-----|----------|-----------|---------------|-------------|
| SSP1-1.9 | C1 | T<1.5 | 5 736 | DEU 1.30 · COL 0.80 · IND 0.88 · KEN/ETH 0.68 |
| SSP1-2.6 | C3 | T<2 | 121 | DEU 1.28 · COL 0.82 · IND 0.90 · KEN/ETH 0.72 |
| SSP2-4.5 | C4 | PARIS_UPDATE | 96 | DEU 1.25 · COL 0.84 · IND 0.91 · KEN/ETH 0.74 |
| SSP3-7.0 | C5 | BASE | 105 | DEU 1.20 · COL 0.86 · IND 0.92 · KEN/ETH 0.78 |
| SSP4-6.0 | C6 | ALTDAM | 191 | DEU 1.18 · COL 0.87 · IND 0.92 · KEN/ETH 0.80 |
| SSP5-8.5 | C8 | OPTIMAL | 79 | DEU 1.15 · COL 0.88 · IND 0.93 · KEN/ETH 0.82 |

> **Note:** SSP1-1.9 carbon prices are extreme (5 736 USD/t global) — shown on a separate
> panel to preserve readability of the other five scenarios.

In [ ]:
# ── Full 6-SSP: REMIND country-specific + UBA×RM + UBA Central ─────────────────
SSP6 = [
    dict(ssp="SSP1-1.9", cat="C1",  key="COEFFICIENT GHG_T<1.5, in USD (WifOR)",
         color="#7b241c",
         mults={"Germany":1.30,"Colombia":0.80,"India":0.88,"Kenya":0.68,"Ethiopia":0.68}),
    dict(ssp="SSP1-2.6", cat="C3",  key="COEFFICIENT GHG_T<2, in USD (WifOR)",
         color="#e74c3c",
         mults={"Germany":1.28,"Colombia":0.82,"India":0.90,"Kenya":0.72,"Ethiopia":0.72}),
    dict(ssp="SSP2-4.5", cat="C4",  key="COEFFICIENT GHG_PARIS_UPDATE, in USD (WifOR)",
         color="#e67e22",
         mults={"Germany":1.25,"Colombia":0.84,"India":0.91,"Kenya":0.74,"Ethiopia":0.74}),
    dict(ssp="SSP3-7.0", cat="C5",  key="COEFFICIENT GHG_BASE, in USD (WifOR)",
         color="#f39c12",
         mults={"Germany":1.20,"Colombia":0.86,"India":0.92,"Kenya":0.78,"Ethiopia":0.78}),
    dict(ssp="SSP4-6.0", cat="C6",  key="COEFFICIENT GHG_ALTDAM, in USD (WifOR)",
         color="#27ae60",
         mults={"Germany":1.18,"Colombia":0.87,"India":0.92,"Kenya":0.80,"Ethiopia":0.80}),
    dict(ssp="SSP5-8.5", cat="C8",  key="COEFFICIENT GHG_OPTIMAL, in USD (WifOR)",
         color="#2980b9",
         mults={"Germany":1.15,"Colombia":0.88,"India":0.93,"Kenya":0.82,"Ethiopia":0.82}),
]

# Load 2030 global coefficients from h5
with pd.HDFStore(str(VF_PATH), "r") as _st:
    _ghg30_full = _st["/coefficient"]

def coeff_global_2030(key):
    return abs(float(_ghg30_full.loc[("2030", key)].iloc[0])) * 1_000

for sc in SSP6:
    sc["global_coeff"] = coeff_global_2030(sc["key"])

# Build full results table
ssp_rows = []
for _, proj in mdf_uba_rm.iterrows():
    row = dict(project=proj.project_id, country=proj.country,
               net_uba_c=proj.net_uba_c,
               av=proj.avoided_tco2e, sc=proj.sc_tco2e, has_av=proj.has_avoided)
    for sc in SSP6:
        mult   = sc["mults"][proj.country]
        c_rm   = sc["global_coeff"] * mult          # REMIND country price
        c_ubarm= UBA_C_USD * mult                    # UBA×RM
        net_rm  = to_musd(proj.avoided_tco2e - proj.sc_tco2e, c_rm)
        net_urm = to_musd(proj.avoided_tco2e - proj.sc_tco2e, c_ubarm)
        k = sc["ssp"]
        row[k+"_c_rm"]    = c_rm
        row[k+"_c_ubarm"] = c_ubarm
        row[k+"_net_rm"]  = net_rm
        row[k+"_net_urm"] = net_urm
        row[k+"_delta_rm"]= net_rm  - proj.net_uba_c
        row[k+"_delta_urm"]=net_urm - proj.net_uba_c
    ssp_rows.append(row)
sdf = pd.DataFrame(ssp_rows)

ssps = [sc["ssp"] for sc in SSP6]

# Print global prices
print("Global 2030 carbon prices (WifOR)  &  REMIND-MAgPIE country-specific prices")
print(f"{'SSP':<12} {'Global':>8}  {'DEU':>8} {'COL':>8} {'IND':>8} {'KEN':>8} {'ETH':>8}")
print("─" * 66)
for sc in SSP6:
    g = sc["global_coeff"]
    m = sc["mults"]
    print(f"{sc['ssp']:<12} {g:>8.1f}  "
          f"{g*m['Germany']:>8.1f} {g*m['Colombia']:>8.1f} "
          f"{g*m['India']:>8.1f} {g*m['Kenya']:>8.1f} {g*m['Ethiopia']:>8.1f}")
print(f"{'UBA Central':<12} {UBA_C_USD:>8.1f}  (uniform across all countries)")

# Print net valuation table (REMIND country-specific)
print(f"\n\nNet valuation — REMIND-MAgPIE country-specific  [MUSD]")
hdr2 = f"{'Project':<12} {'UBA_C':>8}  " + "  ".join(f"{s[:8]:>8}" for s in ssps)
print(hdr2)
print("─" * len(hdr2))
for _, r in sdf.iterrows():
    vals = "  ".join(f"{r[s+'_net_rm']:>8.1f}" for s in ssps)
    print(f"{r.project+'('+r.country[:3]+')':12} {r.net_uba_c:>8.1f}  {vals}")
print("─" * len(hdr2))
tot_vals = "  ".join(f"{sdf[s+'_net_rm'].sum():>8.1f}" for s in ssps)
print(f"{'PORTFOLIO':<12} {sdf.net_uba_c.sum():>8.1f}  {tot_vals}")

# Print UBA×RM table
print(f"\n\nNet valuation — UBA×RM  [MUSD]")
print(hdr2)
print("─" * len(hdr2))
for _, r in sdf.iterrows():
    vals = "  ".join(f"{r[s+'_net_urm']:>8.1f}" for s in ssps)
    print(f"{r.project+'('+r.country[:3]+')':12} {r.net_uba_c:>8.1f}  {vals}")
print("─" * len(hdr2))
tot_vals2 = "  ".join(f"{sdf[s+'_net_urm'].sum():>8.1f}" for s in ssps)
print(f"{'PORTFOLIO':<12} {sdf.net_uba_c.sum():>8.1f}  {tot_vals2}")

In [ ]:
# ── Per-project: REMIND vs UBA×RM vs UBA Central · SSP2-4.5 through SSP5-8.5 ──
# (SSP1-1.9 excluded from this panel — extreme scale, shown separately)
from matplotlib.patches import Patch

PLOT_SSPS = [sc for sc in SSP6 if sc["ssp"] != "SSP1-1.9"]   # 5 scenarios

fig10, axes10 = plt.subplots(3, 3, figsize=(18, 15),
                              gridspec_kw={"hspace":0.62, "wspace":0.35})
fig10.suptitle(
    "Per-Project GHG Valuation — REMIND-MAgPIE Country-Specific vs UBA×RM vs UBA Central\n"
    "SSP1-2.6 → SSP5-8.5  ·  Net MUSD = Avoided − Supply-Chain  ·  Year 2030",
    fontsize=12, fontweight="bold"
)

for pidx, (ax, (_, proj)) in enumerate(zip(axes10.flat, sdf.iterrows())):
    x_s = np.arange(len(PLOT_SSPS))
    _bw = 0.26

    # UBA Central: flat dashed line
    ax.axhline(proj.net_uba_c, color="#2c3e50", lw=1.6, ls="--", zorder=5,
               label=f"UBA Cent.={proj.net_uba_c:.0f}")

    # REMIND country-specific (solid) and UBA×RM (hatched) per scenario
    for xi, sc in enumerate(PLOT_SSPS):
        net_rm  = proj[sc["ssp"]+"_net_rm"]
        net_urm = proj[sc["ssp"]+"_net_urm"]
        ax.bar(xi - _bw/2, net_rm,  width=_bw, color=sc["color"],
               edgecolor="white", lw=0.7,
               label="REMIND" if xi == 0 else "")
        ax.bar(xi + _bw/2, net_urm, width=_bw, color=sc["color"],
               edgecolor="white", lw=0.7, hatch="//", alpha=0.75,
               label="UBA×RM" if xi == 0 else "")
        for val, offset in [(net_rm, -_bw/2), (net_urm, _bw/2)]:
            off = max(abs(val)*0.06, 1.5)
            ax.text(xi + offset, val + (off if val>=0 else -off),
                    f"{val:.0f}", ha="center",
                    va="bottom" if val>=0 else "top", fontsize=7, fontweight="bold")

    ax.axhline(0, color="#888", lw=0.6, ls=":")
    ax.set_xticks(x_s)
    ax.set_xticklabels([sc["ssp"] for sc in PLOT_SSPS], fontsize=7.5, rotation=15, ha="right")
    ax.set_title(
        f"{proj.project} · {proj.country}\n"
        f"{'(avoided ✓)' if proj.has_av else '(no avoided GHG)'}",
        fontsize=8.5, fontweight="bold"
    )
    ax.set_ylabel("Net MUSD", fontsize=8)
    if pidx == 0:
        ax.legend(fontsize=7, loc="lower right", framealpha=0.85)
    ax.spines[["top","right"]].set_visible(False)

legend_p = (
    [Patch(fc=sc["color"], label=sc["ssp"]+" "+sc["cat"]) for sc in PLOT_SSPS] +
    [Patch(fc="#aaa", label="Solid = REMIND cntry-spec"),
     Patch(fc="#aaa", hatch="//", alpha=0.75, label="Hatched = UBA×RM")]
)
fig10.legend(handles=legend_p, loc="lower center", ncol=7,
             fontsize=8.5, framealpha=0.9, bbox_to_anchor=(0.5, 0.01))
plt.savefig(FIG_DIR / "fig_ghg_6ssp_per_project.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved: fig_ghg_6ssp_per_project.png")

In [ ]:
# ── Portfolio fan + Δ heatmap + SSP1-1.9 panel ────────────────────────────────
import matplotlib.gridspec as _gs10

fig11 = plt.figure(figsize=(18, 14))
gs11 = _gs10.GridSpec(3, 3, hspace=0.55, wspace=0.40,
                      top=0.93, bottom=0.06, left=0.07, right=0.97)
ax_port11 = fig11.add_subplot(gs11[0, :2])   # portfolio bars (5 SSPs)
ax_ssp1   = fig11.add_subplot(gs11[0, 2])    # SSP1-1.9 panel
ax_heat11 = fig11.add_subplot(gs11[1, :])    # Δ heatmap (all 6 × 9 projects)
ax_cty11  = fig11.add_subplot(gs11[2, :2])   # net by country
ax_vf11   = fig11.add_subplot(gs11[2, 2])    # country VF comparison

# ── Panel A: portfolio net for 5 SSPs (excl. SSP1-1.9) ────────────────────────
PLOT5 = [sc for sc in SSP6 if sc["ssp"] != "SSP1-1.9"]
x_p = np.arange(len(PLOT5))
_bw11 = 0.28
for xi, sc in enumerate(PLOT5):
    tot_rm  = sdf[sc["ssp"]+"_net_rm"].sum()
    tot_urm = sdf[sc["ssp"]+"_net_urm"].sum()
    ax_port11.bar(xi - _bw11/2, tot_rm,  width=_bw11, color=sc["color"],
                  edgecolor="white", lw=0.8)
    ax_port11.bar(xi + _bw11/2, tot_urm, width=_bw11, color=sc["color"],
                  edgecolor="white", lw=0.8, hatch="//", alpha=0.75)
    for val, offset in [(tot_rm,-_bw11/2),(tot_urm,_bw11/2)]:
        off = max(abs(val)*0.03, 4)
        ax_port11.text(xi+offset, val+(off if val>=0 else -off),
                       f"{val:.0f}", ha="center",
                       va="bottom" if val>=0 else "top",
                       fontsize=8.5, fontweight="bold")
uba_tot = sdf.net_uba_c.sum()
ax_port11.axhline(uba_tot, color="#2c3e50", lw=1.8, ls="--",
                  label=f"UBA Central = {uba_tot:.0f} MUSD")
ax_port11.axhline(0, color="#888", lw=0.7, ls=":")
ax_port11.set_xticks(x_p)
ax_port11.set_xticklabels([sc["ssp"]+"\n"+sc["cat"] for sc in PLOT5], fontsize=9)
ax_port11.set_ylabel("Portfolio Net MUSD", fontsize=9)
ax_port11.set_title("Portfolio Net — REMIND (solid) vs UBA×RM (hatched)  ·  SSP1-2.6 → SSP5-8.5",
                    fontsize=9.5, fontweight="bold")
ax_port11.legend(fontsize=8.5, framealpha=0.9)
ax_port11.spines[["top","right"]].set_visible(False)
from matplotlib.patches import Patch as _P11
ax_port11.add_artist(ax_port11.legend(
    handles=[ax_port11.get_lines()[0],
             _P11(fc="#888", label="REMIND cntry-spec"),
             _P11(fc="#888", hatch="//", alpha=0.75, label="UBA×RM")],
    fontsize=8, loc="lower right", framealpha=0.9))

# ── Panel B: SSP1-1.9 separate (extreme scale) ────────────────────────────────
sc1 = SSP6[0]
proj_labels11 = [f"{r.project}\n({r.country[:3]})" for _, r in sdf.iterrows()]
y11 = np.arange(len(sdf))
ax_ssp1.barh(y11 - 0.18, [sdf.loc[i, sc1["ssp"]+"_net_rm"]  for i in sdf.index],
             height=0.32, color=sc1["color"], edgecolor="white", lw=0.5,
             label="REMIND")
ax_ssp1.barh(y11 + 0.18, [sdf.loc[i, sc1["ssp"]+"_net_urm"] for i in sdf.index],
             height=0.32, color=sc1["color"], hatch="//", alpha=0.75,
             edgecolor="white", lw=0.5, label="UBA×RM")
ax_ssp1.axvline(0, color="#888", lw=0.8, ls=":")
ax_ssp1.set_yticks(y11)
ax_ssp1.set_yticklabels(proj_labels11, fontsize=7.5)
ax_ssp1.set_title("SSP1-1.9 (C1)\n< 1.5°C · global 5 736 USD/t",
                  fontsize=9, fontweight="bold", color=sc1["color"])
ax_ssp1.set_xlabel("Net MUSD", fontsize=8)
ax_ssp1.legend(fontsize=7.5, framealpha=0.9)
ax_ssp1.spines[["top","right"]].set_visible(False)

# ── Panel C: Δ heatmap (REMIND − UBA Central) for all 6 SSPs ─────────────────
delta_mat11 = np.array(
    [[sdf.loc[i, s+"_delta_rm"] for s in [sc["ssp"] for sc in SSP6]]
     for i in sdf.index]
)
vmax11 = np.percentile(np.abs(delta_mat11), 95)  # cap at 95th pct (handles SSP1-1.9)
im11 = ax_heat11.imshow(delta_mat11, aspect="auto", cmap="RdYlGn",
                        vmin=-vmax11, vmax=vmax11)
ax_heat11.set_xticks(range(6))
ax_heat11.set_xticklabels(
    [f"{sc['ssp']}\n{sc['cat']}" for sc in SSP6], fontsize=9)
ax_heat11.set_yticks(range(len(sdf)))
ax_heat11.set_yticklabels(proj_labels11, fontsize=8.5)
ax_heat11.set_title(
    "Δ REMIND-MAgPIE country-specific − UBA Central  [MUSD]  ·  All 6 SSPs × 9 Projects\n"
    "(colour scale capped at 95th percentile; SSP1-1.9 values may exceed scale)",
    fontsize=9.5, fontweight="bold"
)
for r in range(len(sdf)):
    for c_ in range(6):
        val = delta_mat11[r, c_]
        txt = f"{'+'if val>=0 else ''}{val:.0f}"
        ax_heat11.text(c_, r, txt, ha="center", va="center",
                       fontsize=8, fontweight="bold",
                       color="white" if abs(val)>vmax11*0.55 else "#222")
plt.colorbar(im11, ax=ax_heat11, shrink=0.6, label="Δ MUSD vs UBA Central")

# ── Panel D: country net per SSP (REMIND) ─────────────────────────────────────
country_order11 = ["Colombia","Kenya","Ethiopia","India","Germany"]
x_c11 = np.arange(len(country_order11))
_bwc11 = 0.14
for si, sc in enumerate(PLOT5):
    offset = (si - len(PLOT5)/2 + 0.5) * _bwc11
    cvals  = [sdf[sdf.country==c][sc["ssp"]+"_net_rm"].sum() for c in country_order11]
    ax_cty11.bar(x_c11 + offset, cvals, width=_bwc11*0.9,
                 color=sc["color"], edgecolor="white", lw=0.5,
                 label=sc["ssp"])
uba_cvals = [sdf[sdf.country==c]["net_uba_c"].sum() for c in country_order11]
ax_cty11.plot(x_c11, uba_cvals, "o--", color="#2c3e50", lw=1.5,
              markersize=6, label="UBA Central", zorder=5)
ax_cty11.axhline(0, color="#888", lw=0.7, ls=":")
ax_cty11.set_xticks(x_c11)
ax_cty11.set_xticklabels(country_order11, fontsize=9)
ax_cty11.set_ylabel("Net MUSD", fontsize=9)
ax_cty11.set_title("Country-Level Net  ·  REMIND Country-Specific  ·  SSP1-2.6 → SSP5-8.5",
                   fontsize=9.5, fontweight="bold")
ax_cty11.legend(fontsize=7.5, ncol=3, framealpha=0.9, loc="lower right")
ax_cty11.spines[["top","right"]].set_visible(False)

# ── Panel E: country VF spread ─────────────────────────────────────────────────
for ci, country in enumerate(country_order11):
    vfs = [sc["global_coeff"] * sc["mults"][country] for sc in SSP6]
    ax_vf11.plot([sc["ssp"] for sc in SSP6], vfs,
                 "o-", lw=1.5, markersize=6, label=country)
ax_vf11.axhline(UBA_C_USD, color="#2c3e50", lw=1.5, ls="--",
                label=f"UBA ({UBA_C_USD:.0f})")
ax_vf11.set_yscale("log")
ax_vf11.set_ylabel("REMIND country price  [USD/t, log scale]", fontsize=8)
ax_vf11.set_title("REMIND Country Prices per SSP\n(log scale · UBA Central dashed)",
                  fontsize=9.5, fontweight="bold")
ax_vf11.set_xticklabels(ax_vf11.get_xticklabels(), fontsize=7.5, rotation=20, ha="right")
ax_vf11.legend(fontsize=7.5, framealpha=0.9, loc="upper left")
ax_vf11.spines[["top","right"]].set_visible(False)

fig11.suptitle(
    "Full 6-SSP Assessment — REMIND-MAgPIE Country-Specific vs UBA×RM vs UBA Central\n"
    "All 9 portfolio projects  ·  5 countries  ·  Year 2030",
    fontsize=12, fontweight="bold"
)
plt.savefig(FIG_DIR / "fig_ghg_6ssp_portfolio.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved: fig_ghg_6ssp_portfolio.png")

## Current vs Future Avoided CO₂ Valuation — Scatter by Project & SSP

Each project is positioned on an **X–Y axis** where:

- **X** = current avoided CO₂ monetisation · **UBA Central (412.50 USD/t)** · MUSD  
- **Y** = future avoided CO₂ monetisation · **REMIND-MAgPIE country-specific price** · MUSD

The **diagonal y = x** is the no-change reference line.
Points **above** the diagonal: REMIND prices the avoided CO₂ higher than UBA Central → positive future uplift.  
Points **below** the diagonal: REMIND prices it lower → future discount vs current valuation.  

Hospitals (HOSP-1/2/3) report no avoided GHG and appear at the origin in all panels.

In [ ]:
# ── Scatter: current (UBA) vs future (REMIND) avoided CO₂ valuation ───────────
from matplotlib.lines import Line2D

# Per-project: x = avoided × UBA_C, y = avoided × REMIND_country per SSP
SECTOR_STYLE = {
    "HOSP":  dict(marker="X", size=110, color="#95a5a6", zorder=3),
    "HYDRO": dict(marker="o", size=160, color="#2980b9", zorder=5),
    "RAIL":  dict(marker="s", size=140, color="#e67e22", zorder=5),
}

fig12, axes12 = plt.subplots(2, 3, figsize=(17, 11),
                              gridspec_kw={"hspace":0.50, "wspace":0.38})
fig12.suptitle(
    "Current vs Future Avoided CO₂ Valuation — UBA Central (X) vs REMIND-MAgPIE Country-Specific (Y)\n"
    "Each point = one project  ·  Diagonal = no-change line  ·  Above = future uplift  ·  Below = future discount",
    fontsize=12, fontweight="bold"
)

for ax, sc in zip(axes12.flat, SSP6):
    ssp_key = sc["ssp"]

    # Compute per-project avoided-only values
    xs, ys, labels, styles = [], [], [], []
    for _, row in sdf.iterrows():
        x_val = to_musd(row["av"], UBA_C_USD)              # current: UBA
        y_val = to_musd(row["av"], row[ssp_key+"_c_rm"])   # future: REMIND
        xs.append(x_val)
        ys.append(y_val)
        labels.append(row["project"])
        sector = row["project"].split("-")[0]
        styles.append(SECTOR_STYLE[sector])

    # Diagonal reference
    all_vals = xs + ys
    lo = min(v for v in all_vals) * 1.08
    hi = max(v for v in all_vals) * 1.08
    diag = [min(lo, 0), hi]
    ax.plot(diag, diag, color="#555", lw=1.2, ls="--", zorder=1, label="y = x")

    # Shade above/below diagonal
    ax.fill_between(diag, diag, [hi, hi], alpha=0.06, color="#27ae60")
    ax.fill_between(diag, [lo, lo], diag, alpha=0.06, color="#e74c3c")

    # Plot each project
    for x, y, label, st in zip(xs, ys, labels, styles):
        ax.scatter(x, y, marker=st["marker"], s=st["size"],
                   color=st["color"], edgecolors="white", linewidths=0.8,
                   zorder=st["zorder"])
        # Offset label to avoid overlap
        x_off = (hi - lo) * 0.025
        y_off = (hi - lo) * 0.028
        ax.annotate(
            label,
            xy=(x, y),
            xytext=(x + x_off, y + y_off),
            fontsize=7.5, fontweight="bold",
            arrowprops=dict(arrowstyle="-", color="#888", lw=0.6),
        )

    # Delta annotations for projects with avoided GHG
    for x, y, label in zip(xs, ys, labels):
        if x > 0:
            delta_pct = (y - x) / x * 100
            sign = "+" if delta_pct >= 0 else ""
            ax.text(x, y - (hi - lo) * 0.045,
                    f"{sign}{delta_pct:.0f}%",
                    ha="center", va="top", fontsize=7,
                    color="#27ae60" if delta_pct >= 0 else "#e74c3c",
                    fontweight="bold")

    ax.set_xlabel("Current: UBA Central · Avoided CO₂  [MUSD]", fontsize=8.5)
    ax.set_ylabel(f"Future: REMIND {sc['ssp']} · Avoided CO₂  [MUSD]", fontsize=8.5)
    ax.set_title(
        f"{sc['ssp']}  ·  {sc['cat']}  ·  "
        f"Global: {sc['global_coeff']:.0f} USD/t",
        fontsize=9.5, fontweight="bold", color=sc["color"]
    )
    ax.set_xlim(min(lo, 0), hi)
    ax.set_ylim(min(lo, 0), hi)
    ax.spines[["top", "right"]].set_visible(False)

    # UBA Central reference on y-axis
    if sc["global_coeff"] != UBA_C_USD:
        ratio = sc["global_coeff"] / UBA_C_USD
        ax.text(0.02, 0.97,
                f"Global RM / UBA = {ratio:.2f}×",
                transform=ax.transAxes, fontsize=7.5, va="top",
                color=sc["color"], fontweight="bold")

# Legend
legend_handles = [
    Line2D([0],[0], marker="X", color="w", markerfacecolor="#95a5a6",
           markersize=9, label="Hospital (no avoided GHG)"),
    Line2D([0],[0], marker="o", color="w", markerfacecolor="#2980b9",
           markersize=9, label="Hydro"),
    Line2D([0],[0], marker="s", color="w", markerfacecolor="#e67e22",
           markersize=9, label="Rail"),
    Line2D([0],[0], color="#555", ls="--", lw=1.2, label="y = x (no change)"),
]
fig12.legend(handles=legend_handles, loc="lower center", ncol=4,
             fontsize=9, framealpha=0.9, bbox_to_anchor=(0.5, 0.01))
plt.savefig(FIG_DIR / "fig_ghg_scatter_current_vs_future.png", dpi=140, bbox_inches="tight")
plt.show()
print("Saved: fig_ghg_scatter_current_vs_future.png")

In [ ]:
# ── Key finding: UBA Central vs REMIND country prices across 6 SSPs ───────────
from matplotlib.patches import FancyArrowPatch, Rectangle
from matplotlib.lines import Line2D
from matplotlib.colors import TwoSlopeNorm

SSP_ORDER = [
    ("SSP1-1.9", "C1\n<1.5°C"),
    ("SSP1-2.6", "C3\n<2°C"),
    ("SSP2-4.5", "C4\n~2°C Paris"),
    ("SSP4-6.0", "C6\n~3°C"),
    ("SSP3-7.0", "C5\n~3°C baseline"),
    ("SSP5-8.5", "C8\n>3°C"),
]
ssp_keys   = [s[0] for s in SSP_ORDER]
ssp_xticks = [s[1] for s in SSP_ORDER]
x_pos = np.arange(len(SSP_ORDER))

COUNTRY_STYLE = {
    "Germany":   dict(color="#2c3e50", marker="s", lw=2.0, ms=9,  ls="-"),
    "India":     dict(color="#e74c3c", marker="o", lw=2.0, ms=9,  ls="-"),
    "Colombia":  dict(color="#e67e22", marker="^", lw=1.8, ms=8,  ls="-"),
    "Kenya":     dict(color="#27ae60", marker="D", lw=1.8, ms=7,  ls="-"),
    "Ethiopia":  dict(color="#8e44ad", marker="v", lw=1.8, ms=7,  ls="-"),
}

# SSP assumption data for annotation table
SSP_ASSUMPTIONS = [
    dict(
        ssp="SSP1-1.9", cat="C1", warming="< 1.5°C",
        wifor="T<1.5", global_coeff=5736,
        description="Sustainability. Rapid decarbonisation, carbon removal (CDR), net-zero before 2050. Requires immediate economy-wide transformation.",
        rm_basis="R5 mults: DEU 1.30 · COL 0.80 · IND 0.88 · KEN/ETH 0.68",
        color="#7b241c",
    ),
    dict(
        ssp="SSP1-2.6", cat="C3", warming="< 2°C",
        wifor="T<2", global_coeff=121,
        description="Paris Agreement 2°C pathway. Significant mitigation, renewable transition, carbon pricing. Net-zero around 2070.",
        rm_basis="R5 mults: DEU 1.28 · COL 0.82 · IND 0.90 · KEN/ETH 0.72",
        color="#e74c3c",
    ),
    dict(
        ssp="SSP2-4.5", cat="C4", warming="~2–2.5°C",
        wifor="PARIS_UPDATE", global_coeff=96,
        description="Middle of the road. Current policies plus moderate additional measures. Partial decarbonisation of energy sector by 2050.",
        rm_basis="R5 mults: DEU 1.25 · COL 0.84 · IND 0.91 · KEN/ETH 0.74",
        color="#e67e22",
    ),
    dict(
        ssp="SSP4-6.0", cat="C6", warming="~2.7–3.1°C",
        wifor="ALTDAM", global_coeff=191,
        description="Unequal world. High-income countries decarbonise; developing nations lag. Alternative (higher) damage function applied.",
        rm_basis="R5 mults: DEU 1.18 · COL 0.87 · IND 0.92 · KEN/ETH 0.80",
        color="#27ae60",
    ),
    dict(
        ssp="SSP3-7.0", cat="C5", warming="~2.8–3.6°C",
        wifor="BASE", global_coeff=105,
        description="Regional rivalry. Fragmented policies, limited international cooperation. Slow energy transition. Baseline SCC.",
        rm_basis="R5 mults: DEU 1.20 · COL 0.86 · IND 0.92 · KEN/ETH 0.78",
        color="#f39c12",
    ),
    dict(
        ssp="SSP5-8.5", cat="C8", warming="3.3–5.7°C",
        wifor="OPTIMAL", global_coeff=79,
        description="Fossil-fuelled development. Very high emissions, minimal climate policy. Socially optimal SCC under high discount rate.",
        rm_basis="R5 mults: DEU 1.15 · COL 0.88 · IND 0.93 · KEN/ETH 0.82",
        color="#2980b9",
    ),
]

# ── Figure layout: 3 rows ──────────────────────────────────────────────────────
fig13 = plt.figure(figsize=(18, 16))
import matplotlib.gridspec as _gs13
gs13 = _gs13.GridSpec(3, 2, figure=fig13,
                      height_ratios=[3, 3, 2.2],
                      hspace=0.55, wspace=0.35,
                      top=0.94, bottom=0.03, left=0.06, right=0.97)
ax_main  = fig13.add_subplot(gs13[0:2, 0])   # log-scale chart spans rows 0–1
ax_ratio = fig13.add_subplot(gs13[0:2, 1])   # heatmap spans rows 0–1
ax_tbl   = fig13.add_subplot(gs13[2, :])     # full-width assumption table

# ── Panel A: log-scale carbon price chart ─────────────────────────────────────
ax = ax_main
ax.set_yscale("log")
_x_shade = [-0.5, len(SSP_ORDER) - 0.5]
ax.fill_between(_x_shade, [10, 10], [UBA_C_USD, UBA_C_USD],
                color="#f9ebea", alpha=0.7, zorder=0)
ax.fill_between(_x_shade, [UBA_C_USD, UBA_C_USD], [12000, 12000],
                color="#eafaf1", alpha=0.5, zorder=0)
sc_lookup = {sc["ssp"]: sc for sc in SSP6}
for country, st in COUNTRY_STYLE.items():
    prices = [sc_lookup[ssp]["global_coeff"] * sc_lookup[ssp]["mults"][country]
               for ssp in ssp_keys]
    ax.plot(x_pos, prices, color=st["color"], marker=st["marker"],
            lw=st["lw"], markersize=st["ms"], ls=st["ls"],
            label=country, zorder=4)
    ax.text(x_pos[-1] + 0.08, prices[-1], f"  {country}",
            va="center", fontsize=8.5, color=st["color"], fontweight="bold")
ax.axhline(UBA_C_USD, color="#1a1a1a", lw=2.2, ls="--", zorder=5,
           label=f"UBA Central  (412.50 USD/t)")
ax.text(-0.45, UBA_C_USD * 1.12, "UBA Central\n412.50 USD/t",
        fontsize=8.5, fontweight="bold", color="#1a1a1a", va="bottom")
ax.text(5.35, 600,  "UBA exceeds REMIND", fontsize=8, color="#c0392b",
        va="center", ha="left", style="italic",
        bbox=dict(fc="#fdecea", ec="none", alpha=0.8))
ax.text(5.35, 2200, "REMIND exceeds UBA", fontsize=8, color="#1e8449",
        va="center", ha="left", style="italic",
        bbox=dict(fc="#eafaf1", ec="none", alpha=0.8))
ssp1_deu = sc_lookup["SSP1-1.9"]["global_coeff"] * sc_lookup["SSP1-1.9"]["mults"]["Germany"]
ax.annotate(
    f"SSP1-1.9 / DEU\n{ssp1_deu:.0f} USD/t  ({ssp1_deu/UBA_C_USD:.1f}× UBA)",
    xy=(0, ssp1_deu), xytext=(0.7, ssp1_deu * 0.55),
    fontsize=8, color="#2c3e50", fontweight="bold",
    arrowprops=dict(arrowstyle="->", color="#2c3e50", lw=1.2),
)
ax.set_xticks(x_pos)
ax.set_xticklabels(ssp_xticks, fontsize=9.5)
ax.set_xlim(-0.55, len(SSP_ORDER) - 0.45)
ax.set_ylim(40, 12000)
ax.set_ylabel("Carbon price  [USD / tCO₂e · log scale]", fontsize=10)
ax.set_title("REMIND-MAgPIE Country Prices vs UBA Central · Year 2030",
             fontsize=11, fontweight="bold")
ax.legend(fontsize=8.5, loc="upper right", framealpha=0.9)
ax.spines[["top", "right"]].set_visible(False)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f}"))

# ── Panel B: ratio heatmap ─────────────────────────────────────────────────────
countries_ord = ["Germany", "India", "Colombia", "Kenya", "Ethiopia"]
ratio_mat = np.array([
    [sc_lookup[ssp]["global_coeff"] * sc_lookup[ssp]["mults"][c] / UBA_C_USD
     for ssp in ssp_keys]
    for c in countries_ord
])
norm_r = TwoSlopeNorm(vmin=ratio_mat.min(), vcenter=1.0, vmax=ratio_mat.max())
im_r = ax_ratio.imshow(ratio_mat, aspect="auto", cmap="RdYlGn", norm=norm_r)
ax_ratio.set_xticks(range(len(SSP_ORDER)))
ax_ratio.set_xticklabels([s[0].replace("SSP","") for s in SSP_ORDER],
                          fontsize=8.5, rotation=30, ha="right")
ax_ratio.set_yticks(range(len(countries_ord)))
ax_ratio.set_yticklabels(countries_ord, fontsize=9)
ax_ratio.set_title("REMIND / UBA ratio\n(green > 1 = REMIND > UBA)",
                   fontsize=10, fontweight="bold")
for r in range(len(countries_ord)):
    for c_ in range(len(SSP_ORDER)):
        v = ratio_mat[r, c_]
        txt = f"{v:.1f}×" if v < 10 else f"{v:.0f}×"
        ax_ratio.text(c_, r, txt, ha="center", va="center",
                      fontsize=8.5, fontweight="bold",
                      color="white" if (v > 8 or v < 0.25) else "#222")
plt.colorbar(im_r, ax=ax_ratio, shrink=0.75,
             label="REMIND / UBA Central", format="%.1f×")

# ── Panel C: SSP assumption table ─────────────────────────────────────────────
ax_tbl.axis("off")
col_headers = ["SSP", "Cat", "Warming", "WifOR key",
               "Global\n2030 (USD/t)", "Scenario description",
               "REMIND-MAgPIE R5 multipliers"]
col_w = [0.085, 0.038, 0.068, 0.075, 0.075, 0.38, 0.27]
row_h = 0.135
n_rows = len(SSP_ASSUMPTIONS)
x_starts = [sum(col_w[:i]) for i in range(len(col_w))]

# Header row
for j, (hdr, xs, cw) in enumerate(zip(col_headers, x_starts, col_w)):
    ax_tbl.text(xs + cw/2, 1.0, hdr,
                ha="center", va="bottom", fontsize=8.5, fontweight="bold",
                transform=ax_tbl.transAxes,
                bbox=dict(fc="#2c3e50", ec="none", pad=3) if j==0
                     else dict(fc="#34495e", ec="none", pad=3),
                color="white")

# Data rows
for i, sc_a in enumerate(SSP_ASSUMPTIONS):
    y_row = 1.0 - (i + 1) * row_h
    row_bg = "#fdfefe" if i % 2 == 0 else "#f2f3f4"
    ax_tbl.add_patch(Rectangle((0, y_row), 1.0, row_h,
                                transform=ax_tbl.transAxes,
                                fc=row_bg, ec="#ddd", lw=0.5, zorder=0))
    row_vals = [
        sc_a["ssp"], sc_a["cat"], sc_a["warming"],
        sc_a["wifor"], f"{sc_a['global_coeff']:,}",
        sc_a["description"], sc_a["rm_basis"],
    ]
    for j, (val, xs, cw) in enumerate(zip(row_vals, x_starts, col_w)):
        is_ssp_col = (j == 0)
        ax_tbl.text(
            xs + (0.008 if j > 0 else cw/2),
            y_row + row_h * 0.5,
            val,
            ha="center" if is_ssp_col else "left",
            va="center", fontsize=7.8,
            transform=ax_tbl.transAxes,
            color=sc_a["color"] if is_ssp_col else "#222",
            fontweight="bold" if is_ssp_col else "normal",
            wrap=True,
        )

ax_tbl.set_title("SSP Scenario Assumptions — IPCC AR6 WG3 / WifOR / REMIND-MAgPIE 2.1",
                 fontsize=10, fontweight="bold", pad=28)

fig13.suptitle(
    "UBA Central already prices avoided CO₂ above every IPCC ≥ 2°C pathway\n"
    "Only SSP1-1.9 (<1.5°C) generates REMIND-MAgPIE carbon prices that exceed UBA",
    fontsize=12, fontweight="bold"
)
plt.savefig(FIG_DIR / "fig_ghg_uba_vs_remind_finding.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved: fig_ghg_uba_vs_remind_finding.png")

## Methodology Notes

| Item | Detail |
|------|--------|
| **Avoided GHG source** | Self-reported by project promoters; extracted from Greenings framework indicator `E2.Q8W.A` (prior year) or `E2.Q8W.B` (first operational year) |
| **Tier-0 computation** | `tier0_impact(invest_usd, sector_code, region, database="exiobase")` — direct CAPEX distribution across 8 supplying sectors × Exiobase 3.8.1 GHG intensities |
| **Tier-1 computation** | `tier1_impact(...)` — one Leontief round: `A × y₀`, sourcing-region trade shares from calibrated OECD TiVA fallback |
| **Hospitals** | No avoided GHG reported (not a GHG-reduction intervention); SC footprint is still computed for completeness |
| **Unit mismatch** | Avoided GHG is an **annual flow** (tCO₂e/yr); SC GHG is a **one-time stock** tied to CAPEX. Charts show both for structural comparison; net delta should be interpreted in that context |
| **WifOR GHG value factors** | `2024-11-18_formatted_MonGHG_my.h5`; 11 scenarios, globally uniform per scenario; 2019 baseline: BASE = 63.70, PARIS_UPDATE = 57.03, T<2 = 71.20 USD/tCO₂e |
| **WifOR 2030 coefficients** | BASE = 104.73, PARIS_UPDATE = 96.20, T<2 = 121.05, T<1.5 = 5 736.10 USD/tCO₂e (globally uniform) |
| **UBA Methodological Convention 4.0** | `01_uba4_ghg.csv`; CO₂_CO₂eq, 2030: Central PRTP=1% = 375 EUR₂₀₂₅/t × 1.10 = 412.50 USD/t; High PRTP=0% = 1 050 EUR₂₀₂₅/t × 1.10 = 1 155 USD/t |
| **REMIND-MAgPIE R5 multipliers** | Applied to WifOR global 2030 coefficients to derive country-specific 2030 carbon prices. Source: IPCC AR6 WG3 Ch. 3 / Annex III regional shadow carbon price data from REMIND-MAgPIE 2.1. Multipliers: R5OECD+EU (DEU) = 1.28×, R5LAM (COL) = 0.82×, R5ASIA (IND) = 0.90×, R5MAF (KEN, ETH) = 0.72× |
| **Monetisation formula** | `to_musd(tco2e, coeff) = tco2e × coeff / 1 000 000` where `coeff` is USD/tCO₂e |